# Costruzione del panel

Questo notebook costruisce il panel di startup a partire dai CSV grezzi di
**PitchBook** e documenta ogni passaggio. È la traduzione in Python di due
script R scritti da terzi (`src/RCode/1_Arrange_DB.R`, 1257 righe, e
`src/RCode/2_Arrange_Final.R`, 269 righe), più i due passaggi di
post-elaborazione che li seguivano.

**L'obiettivo di questa fase è la riproduzione fedele, bug compresi.** Ogni
difetto noto dell'R viene riprodotto per default e sta dietro a un flag
`fix_*` in `PanelConfig` che vale `False`, dove `False` significa «fai quello
che faceva l'R». Correggerli è una fase successiva: prima si dimostra di aver
tradotto bene, poi si cambia.

Il criterio di successo non è dichiarativo. Gli script R salvavano dei file
intermedi, e quei file sono disponibili in `data/reference/`: ogni stadio viene
confrontato **colonna per colonna, riga per riga** con il file che deve
riprodurre. Sei confronti indipendenti, allineati per chiave e non per
posizione.


## Come si legge e si esegue

Ogni stadio ha tre parti:

1. una spiegazione di **cosa fa** e a quali righe dell'R corrisponde;
2. le **trappole**: i punti in cui una traduzione plausibile divergerebbe in
   silenzio, con il numero di righe che sbaglierebbe;
3. il **codice** che lo esegue, un'**ispezione** del risultato e, dove esiste
   un file di riferimento, il **rapporto di verifica**.

Gli stadi comunicano tramite file parquet in `data/interim/`, non tramite
memoria. Questo ha due conseguenze pratiche: si può **riprendere da qualsiasi
stadio** senza rifare i precedenti, e si può **riavviare il kernel** in
qualsiasi momento senza perdere niente. Se un checkpoint segnala un problema,
si corregge il codice, si riesegue solo quello stadio e si riverifica.

Niente in questo notebook scrive in `data/raw/` o in `data/reference/`: sono
gli input e la verità di riferimento, e restano intatti. Il panel finito
finisce in `data/interim/panel.csv.gz`.

I rapporti di verifica girano in un **sottoprocesso**. Non è un vezzo: leggere
`db_master_2.csv` come testo occupa qualche gigabyte, e tenerlo nel kernel per
il resto della sessione fa saltare la memoria a metà pipeline. Per lo stesso
motivo le celle di ispezione rileggono da parquet solo le colonne che servono e
liberano le variabili quando ha finito: il picco della pipeline è di circa
5,6 GB, e il margine non è grande.


In [ ]:
import gc
import subprocess
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.panel import (
    stage5_final,
    stage6_panel,
    stage7_competitors,
)
from src.panel.config import BUG_FLAGS, PanelConfig
from src.panel.expansions import expand_team
from src.panel.io import COMPANY_DATE_COLUMNS, load_europe, read_raw, to_num
from src.panel.rutils import (
    R_NA,
    R_NA_INF,
    R_NA_NAN,
    as_na,
    coalesce_first_last,
    parse_date_r,
    r_case_when,
    r_if_else,
    r_seq,
    scale_r,
    tail_na_omit,
)
from src.panel.validate import CHECKPOINTS

cfg = PanelConfig()

pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(40)

attive = cfg.active_fixes()
print("correzioni attive:", list(attive) if attive else "nessuna (comportamento R)")
print("flag disponibili :", len(BUG_FLAGS))
print("input grezzi     :", cfg.raw_dir)
print("riferimenti      :", cfg.ref_dir)
print("output di stadio :", cfg.interim_dir)


### Due funzioni di servizio

`verifica` lancia un checkpoint in un sottoprocesso e stampa il rapporto.
`carica` rilegge l'output di uno stadio da parquet, opzionalmente solo alcune
colonne, per ispezionarlo senza tenersi in memoria tutto il resto.


In [ ]:
def verifica(lettera: str) -> None:
    """Esegue un checkpoint in un sottoprocesso e ne stampa il rapporto."""
    cp = CHECKPOINTS[lettera]
    print(f"checkpoint {lettera}: {cp.interim}  contro  {cp.reference}")
    print(f"righe attese: {cp.expect_rows:,}   chiave: {cp.key}\n")
    esito = subprocess.run(
        [sys.executable, "-m", "src.panel.validate", "--checkpoint", lettera],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(esito.stdout or esito.stderr)


def carica(nome: str, colonne: list[str] | None = None) -> pl.DataFrame:
    """Rilegge l'output di uno stadio, proiettato alle colonne richieste."""
    return pl.read_parquet(cfg.interim(f"{nome}.parquet"), columns=colonne)


def confronta(nome: str) -> None:
    """Confronta l'output di uno stadio con la base congelata, in sottoprocesso.

    Piu' forte dei checkpoint e complementare: copre **tutte** le colonne,
    comprese quelle che un checkpoint dichiara e quindi non confronta mai (le
    sei `_Est`, `TR_D`, `StageBlock`, le sei competitor). Da una parte e
    dall'altra ci sono due parquet tipizzati: niente e' confrontato come testo
    e niente e' scusato.
    """
    print(f"{nome}: data/interim  contro  data/baseline\n")
    esito = subprocess.run(
        [sys.executable, "-m", "src.panel.validate", "--baseline", nome],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(esito.stdout or esito.stderr)


## Panoramica: sette stadi, sei checkpoint

| stadio | cosa aggiunge | tabelle grezze lette | checkpoint |
|---|---|---|---|
| 1 | sezione trasversale delle aziende e **scheletro del panel** | `Company`, `CompanyAffiliateRelation` | — |
| 2a | tabella **persona-azienda**: istruzione, esperienza, permanenza | `CompanyBoardTeamRelation`, `Person`, `PersonEducationRelation`, `PersonPositionRelation` | **A** |
| 2b | le colonne di **team** anno per anno | — | — |
| 3a | le colonne **competitor** (statiche) | `CompanySimilarRelation` | **B** |
| 3b | dipendenti, financials, news | `CompanyEmployeeHistoryRelation`, `CompanyFinancialRelation`, `CompanyNewsRelation` | — |
| 4 | **deal e investitori** | `Deal`, `DealInvestorRelation`, `Investor` | — |
| 5 | cumulate, `GrowthStage`, attributi del CEO, selezione colonne | — | **C**, **D** |
| 6 | raggruppamento degli stadi e **troncamento** all'uscita | — | **E** |
| 7 | competitor **temporizzati** anno per anno | `CompanySimilarRelation`, `Company` | **F** |

Gli stadi 1–5 traducono l'R. Gli stadi 6 e 7 traducono due passaggi che
venivano dopo: il primo non aveva codice sopravvissuto e le sue regole sono
state ricostruite dal file che produceva, il secondo veniva da un notebook
separato.

### Il database SQLite dell'R non serve

Gli script R caricavano 51 CSV in un database SQLite temporaneo e vi accedevano
per **indice posizionale** (`tbl[1]`, `tbl[17]`, ...). L'unica informazione che
quel database forniva era la corrispondenza fra indice e tabella, che è
l'ordine alfabetico dei file:

| idx | tabella | idx | tabella |
|----:|---|----:|---|
| 1 | `Company` | 19 | `Deal` |
| 2 | `CompanyAffiliateRelation` | 22 | `DealInvestorRelation` |
| 3 | `CompanyBoardTeamRelation` | 32 | `Investor` |
| 6 | `CompanyEmployeeHistoryRelation` | 43 | `Person` |
| 8 | `CompanyFinancialRelation` | 48 | `PersonEducationRelation` |
| 14 | `CompanyNewsRelation` | 49 | `PersonPositionRelation` |
| 17 | `CompanySimilarRelation` | | |

La versione Python legge i CSV direttamente, per nome.


## Passo 0 — l'estrazione è quella giusta?

I file in `data/reference/` sono la verità di riferimento. Se i CSV grezzi non
sono lo stesso *vintage* — la stessa data di scarico da PitchBook — nessun
checkpoint può passare, e il problema non sarebbe nel codice.

Il controllo costa pochi secondi e va fatto **prima** di tutto il resto:
confronta il numero di aziende con `YearFounded > 1999` (deve essere
esattamente 116.920) e verifica che nessuna azienda o coppia
azienda-persona presente nei riferimenti manchi dai grezzi.


In [ ]:
print(
    subprocess.run(
        [sys.executable, "scripts/check_extraction.py", "data/raw/pitchbook"],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    ).stdout
)


---
## Stadio 1 — aziende, affiliate e scheletro del panel

**Corrisponde a** `1_Arrange_DB.R:37-238`.

Legge `Company.csv` (134.355 aziende), ne tiene 39 colonne, e:

1. **sostituisce i placeholder di missing con NA veri.** L'R lo fa con un ciclo
   su tutte le colonne che cerca `""`, `"NA"`, `"N/A"`, `"NULL"`, `"NaN"`. Il
   momento in cui questo ciclo gira conta, come si vede subito sotto.
2. **crea quattro flag di presenza** — `Website_d`, `Linkedin`, `Facebook`,
   `Twitter` — come `nchar(x) > 0`.
3. **parsa cinque colonne data** e costruisce `FiscalDate` da `FiscalPeriod`
   (`"TTM 4Q2024"` → quarto trimestre → mese 12 → `2024-12-30`).
4. **conta gli affiliati** da `CompanyAffiliateRelation`: quanti in totale,
   quanti genitori, sorelle, controllate.
5. **costruisce lo scheletro del panel**: per ogni azienda una riga per ogni
   anno da `YearFounded` fino a `MaxYear`, dove `MaxYear` è l'anno più recente
   fra sei date disponibili. È qui che il dataset passa da una riga per azienda
   a una riga per azienda-anno.
6. **aggancia le variabili che variano nel tempo** allo scheletro, con cinque
   join su `(azienda, anno)`: stato di finanziamento, stato di business, stato
   di proprietà, ultima valutazione nota, e i sei dati di bilancio.
7. **filtra `YearFounded > 1999`**, che porta le aziende a 116.920.

**Produce tre file.** `db_master_1_v1.parquet` è la sezione trasversale,
`db_master_2_skeleton.parquet` lo scheletro del panel, e `db1.parquet` contiene
solo `CompanyID` e `YearFounded` **non filtrati**: serve perché alla riga 448
l'R aggancia `YearFounded` alla tabella del team prendendolo dalla versione non
filtrata, e usare quella filtrata farebbe sparire le persone delle aziende più
vecchie.


### Trappole

**Il momento della sostituzione NA.** `nchar(x) > 0` gira *dopo* la
sostituzione, quindi un URL mancante non dà `False` ma **NA**. Nel riferimento
`Website_d` vale infatti solo `True` oppure vuoto, mai `False`. Tradurlo come
«se manca allora False» sembra innocuo e cambia 9.664 righe.

**`seq()` in R conta all'indietro.** Se `MaxYear < YearFounded` — succede quando
una data è sporca — `seq(2010, 2008)` produce `2010, 2009, 2008`, cioè `Delta`
**negativi**. È il bug **B6**: 245 righe su 106 aziende. Lo riproduciamo
generando un intervallo decrescente; il flag `fix_negative_delta` emette invece
solo l'anno di fondazione.

**`pmax(..., na.rm = TRUE)`** su sei anni: se sono tutti mancanti il risultato è
NA e l'azienda viene scartata dallo scheletro.


### 1.1 — leggere `Company.csv` e uniformare i valori mancanti

`Company.csv` ha 134.355 righe, una per azienda. Ne leggiamo 39 colonne
(`1_Arrange_DB.R:47-56`), **tutte come testo**: il cast lo facciamo noi colonna
per colonna, perché l'inferenza automatica sbaglierebbe `CompanyID` — che è una
stringa tipo `"100020-70"` — e romperebbe i join senza dire niente.

`as_na` riproduce il ciclo dell'R alla riga 43: sostituisce con un nullo vero i
cinque segnaposto `""`, `"NA"`, `"N/A"`, `"NULL"`, `"NaN"`. **Il momento in cui
gira conta** (T1): tutto ciò che viene dopo vede nulli, non stringhe vuote.

Nota sul set di segnaposto: la riga 43 include `"NaN"`, la riga 113 — quella
degli affiliati, blocco 1.6 — no. È una differenza reale fra due righe dello
stesso script, e la riproduciamo invece di uniformarla.


In [ ]:
# 1_Arrange_DB.R:47-56 — le 39 colonne tenute da Company.csv.
DB1_COLUMNS = [
    "CompanyID", "CompanyName", "CompanyLegalName", "Description", "Keywords",
    "HQCity", "HQCountry", "HQPostCode", "Website", "YearFounded",
    "CompanyFinancingStatus", "CompanyFinancingStatusDate", "BusinessStatus",
    "BusinessStatusDate", "OwnershipStatus", "OwnershipStatusDate",
    "ParentCompanyID", "PrimaryIndustrySector", "PrimaryIndustryGroup",
    "ActiveInvestors", "FormerInvestors", "FiscalPeriod", "Revenue",
    "GrossProfit", "NetIncome", "EnterpriseValue", "EBITDA", "EBIT",
    "FirstFinancingDate", "FirstFinancingSize", "FirstFinancingValuation",
    "FirstFinancingDealType", "FirstFinancingDebt", "FirstFinancingDebtSize",
    "LastKnownValuation", "LastKnownValuationDate", "FacebookProfileURL",
    "TwitterProfileURL", "LinkedInProfileURL",
]

company = as_na(read_raw(cfg, "Company", DB1_COLUMNS), R_NA_NAN)

print(f"{company.height:,} aziende x {company.width} colonne")
company.select("CompanyID", "CompanyName", "YearFounded", "HQCountry", "Website").head(4)


### 1.2 — i quattro flag di presenza

`Website_d`, `Linkedin`, `Facebook`, `Twitter` nascono da `nchar(x) > 0`.

Siccome girano **dopo** il blocco precedente, un URL mancante è già un nullo e
`nchar(NA) > 0` in R vale `NA`: il flag risulta **nullo, non `False`** (T1).
Nel riferimento infatti `Website_d` vale `True` su 107.256 aziende e nullo su
9.664, e `False` mai. Tradurlo come «se manca allora `False`» cambierebbe
9.664 righe.


In [ ]:
company = company.with_columns(
    pl.col("Website").str.len_chars().gt(0).alias("Website_d"),
    pl.col("LinkedInProfileURL").str.len_chars().gt(0).alias("Linkedin"),
    pl.col("FacebookProfileURL").str.len_chars().gt(0).alias("Facebook"),
    pl.col("TwitterProfileURL").str.len_chars().gt(0).alias("Twitter"),
)

company.select("Website_d", "Linkedin", "Facebook", "Twitter").null_count()


### 1.3 — le date, e i numeri

Cinque colonne data passano da `parse_date_r`, che riproduce la regola a due
rami dell'R (`R:65-84`, voce T4):

| lunghezza della stringa | formato |
|---|---|
| 10 caratteri | `%m/%d/%Y` |
| 8 caratteri | mese/giorno/anno con `cutoff_2000 = 24`: `00`–`24` → 2000-2024, `25`–`99` → 1925-1999 |
| qualsiasi altra | **nullo** |

*In questa estrazione tutte le date di `Company.csv` hanno 10 caratteri*, quindi
il secondo ramo non scatta mai: la regola è latente. La teniamo perché
un'estrazione futura potrebbe attivarla, e perché la stessa primitiva serve
alla fase 7, dove decide se un concorrente è vivo.

Nello stesso blocco portiamo a numero `YearFounded` e i sette importi.


In [ ]:
FINANCIAL_COLUMNS = ["Revenue", "GrossProfit", "NetIncome", "EnterpriseValue", "EBITDA", "EBIT"]

company = company.with_columns(
    *[parse_date_r(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    pl.col("YearFounded").cast(pl.Int64, strict=False),
    *[to_num(c) for c in [*FINANCIAL_COLUMNS, "LastKnownValuation"]],
)

company.select("YearFounded", *COMPANY_DATE_COLUMNS).head(4)


### 1.4 — `FiscalDate` da `FiscalPeriod`

`FiscalPeriod` è una stringa tipo `"TTM 4Q2024"`: quarto trimestre del 2024.
L'R ne ricava il trimestre, lo mappa a un mese (1→3, 2→6, 3→9, 4→12) e
costruisce `Year-Month-30`.

Due dettagli dell'originale (T3): `sub()` senza corrispondenza **restituisce la
stringa intera**, non un nullo, e `Month` diventa nullo solo perché nessun ramo
del `case_when` riconosce quel valore. L'effetto finale è che un
`FiscalPeriod` illeggibile dà una data nulla, ed è quello che riproduciamo con
l'estrazione via regex.

Il giorno 30 è arbitrario ma innocuo: di questa data useremo **solo l'anno**.


In [ ]:
trimestre = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
anno_fiscale = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)

company = company.with_columns(pl.date(anno_fiscale, trimestre * 3, 30).alias("FiscalDate"))

print(company["FiscalPeriod"].value_counts(sort=True).head(3))
company.select("FiscalPeriod", "FiscalDate").drop_nulls().head(3)


### 1.5 — salvare `db1`, la versione **non filtrata**

Due sole colonne, `CompanyID` e `YearFounded`, senza il filtro
`YearFounded > 1999` che applicheremo nel blocco 1.10.

Serve perché alla riga 448 l'R aggancia `YearFounded` alla tabella del team
prendendolo **da qui**, non da `db_master_1` che a quel punto è già filtrato
(voce T5). Usare quello filtrato farebbe sparire le persone delle aziende
fondate prima del 2000 e cambierebbe `db3`, cioè il checkpoint A.


In [ ]:
company.select("CompanyID", "YearFounded").write_parquet(cfg.interim("db1.parquet"))
print(f'db1.parquet: {company.height:,} aziende, senza filtro sull\'anno di fondazione')


### 1.6 — gli affiliati

`CompanyAffiliateRelation` ha una riga per legame societario, con un
`AffiliateType` che vale `Sister`, `Subsidiary` o `Parent`. Contiamo quanti
legami ha ogni azienda, e di che tipo.

Qui il set di segnaposto è `R_NA`, **senza `"NaN"`**: è la differenza fra la
riga 113 e la riga 43 dell'R di cui si parlava nel blocco 1.1. Su queste due
colonne non cambia nulla in pratica, ma è riprodotta perché è reale.


In [ ]:
affiliati = as_na(
    read_raw(cfg, "CompanyAffiliateRelation", ["CompanyID", "AffiliateType"]), R_NA
).group_by("CompanyID").agg(
    pl.len().alias("N_Affiliated"),
    pl.col("AffiliateType").eq("Parent").sum().alias("N_Parent"),
    pl.col("AffiliateType").eq("Sister").sum().alias("N_Sister"),
    pl.col("AffiliateType").eq("Subsidiary").sum().alias("N_Subsidiary"),
)

print(f"{affiliati.height:,} aziende con almeno un legame societario")
affiliati.sort("N_Affiliated", descending=True).head(4)


### 1.7 — agganciare gli affiliati e derivare i flag `Has_*`

Left join, poi i conteggi mancanti diventano **0** (un'azienda senza legami ne
ha zero, non un numero ignoto) e da lì i quattro booleani.

`Affiliated` è `N_Affiliated > 0`: è **lo stesso dato in due forme** (voce X3).
`Has_Parent`, `Has_Sister`, `Has_Subsidiary` sono interi 0/1 e non booleani,
perché nell'R nascono da `as.integer(...)`; il riferimento li ha così.


In [ ]:
db_master_1 = (
    company.join(affiliati, on="CompanyID", how="left")
    .with_columns(pl.col("N_Affiliated", "N_Parent", "N_Sister", "N_Subsidiary").fill_null(0))
    .with_columns(
        pl.col("N_Affiliated").gt(0).alias("Affiliated"),
        pl.col("N_Parent").gt(0).cast(pl.Int64).alias("Has_Parent"),
        pl.col("N_Sister").gt(0).cast(pl.Int64).alias("Has_Sister"),
        pl.col("N_Subsidiary").gt(0).cast(pl.Int64).alias("Has_Subsidiary"),
    )
)

db_master_1.select(
    "N_Affiliated", "Affiliated", "N_Parent", "Has_Parent", "Has_Sister", "Has_Subsidiary"
).head(4)


### 1.8 — `MaxYear` e lo scheletro del panel

**È il blocco in cui il dataset cambia forma**: da una riga per azienda a una
riga per azienda-anno.

`MaxYear` è l'anno più recente fra le sei date disponibili (le cinque del
blocco 1.3 più `FiscalDate`). `pmax(..., na.rm = TRUE)` vale nullo solo se
mancano **tutte** e sei, e in quel caso l'azienda viene scartata: è il filtro
che decide chi entra nel panel (T2).

Poi `r_seq(YearFounded, MaxYear)` genera gli anni. La primitiva vive in
`rutils` perché riproduce una semantica R: `seq()` è inclusivo **e conta
all'indietro** quando l'estremo finale precede quello iniziale. Dove una data
sporca lascia `MaxYear` prima di `YearFounded` la sequenza torna indietro e
produce `Delta` negativi — anni-azienda precedenti alla fondazione. È il
**bug B6**, `fix_negative_delta`: 245 righe su 106 aziende.

`Delta` è l'età dell'azienda in quell'anno, e diventerà `Age` alla fase 5.


In [ ]:
anni_disponibili = [*COMPANY_DATE_COLUMNS, "FiscalDate"]

vita = (
    db_master_1.select(["CompanyID", "YearFounded", *anni_disponibili])
    .with_columns(
        pl.max_horizontal([pl.col(c).dt.year() for c in anni_disponibili]).alias("MaxYear")
    )
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
    .select("CompanyID", "YearFounded", "MaxYear")
)
print(f"aziende con anno di fondazione e MaxYear: {vita.height:,} su {db_master_1.height:,}")

if cfg.fix_negative_delta:
    # Con la correzione attiva: una sola riga, l'anno di fondazione.
    anni = pl.int_ranges(pl.col("YearFounded"), pl.col("YearFounded") + 1)
    anni = pl.when(pl.col("MaxYear") < pl.col("YearFounded")).then(anni).otherwise(
        r_seq(pl.col("YearFounded"), pl.col("MaxYear"))
    )
else:
    anni = r_seq(pl.col("YearFounded"), pl.col("MaxYear"))  # comportamento R, bug B6

scheletro = (
    vita.with_columns(anni.alias("Year_Delta"))
    .explode("Year_Delta", empty_as_null=True)
    .with_columns((pl.col("Year_Delta") - pl.col("YearFounded")).alias("Delta"))
    .select("CompanyID", "YearFounded", "Year_Delta", "Delta")
)
print(f"scheletro: {scheletro.height:,} righe azienda-anno")


### 1.9 — le variabili che variano nel tempo

Cinque left join su `(CompanyID, anno)`. Ogni variabile porta con sé la
**propria** data, e viene agganciata **solo all'anno di quella data**:

| variabile | anno usato per l'aggancio |
|---|---|
| `CompanyFinancingStatus` | anno di `CompanyFinancingStatusDate` |
| `BusinessStatus` | anno di `BusinessStatusDate` |
| `OwnershipStatus` | anno di `OwnershipStatusDate` |
| `LastKnownValuation` | anno di `LastKnownValuationDate` |
| i sei dati di bilancio | anno di `FiscalDate` |

Conseguenza da tenere a mente per la fase 5: **queste colonne sono nulle su
quasi tutte le righe del panel**, perché ciascuna compare in un solo anno per
azienda. `OwnershipStatus` in particolare è lo stato *attuale* dell'azienda
proiettato sull'anno della sua data, ed entra nella cascata che definisce
`GrowthStage` (voce M1).


In [ ]:
con_anni = db_master_1.with_columns(
    pl.col("CompanyFinancingStatusDate").dt.year().alias("_anno_fin"),
    pl.col("BusinessStatusDate").dt.year().alias("_anno_bus"),
    pl.col("OwnershipStatusDate").dt.year().alias("_anno_own"),
    pl.col("LastKnownValuationDate").dt.year().alias("_anno_val"),
    pl.col("FiscalDate").dt.year().alias("_anno_fis"),
)

for colonna_anno, colonne in (
    ("_anno_fin", ["CompanyFinancingStatus"]),
    ("_anno_bus", ["BusinessStatus"]),
    ("_anno_own", ["OwnershipStatus"]),
    ("_anno_val", ["LastKnownValuation"]),
    ("_anno_fis", FINANCIAL_COLUMNS),
):
    scheletro = scheletro.join(
        con_anni.select(["CompanyID", colonna_anno, *colonne]).drop_nulls(colonna_anno),
        left_on=["CompanyID", "Year_Delta"],
        right_on=["CompanyID", colonna_anno],
        how="left",
    )

print(f"scheletro: {scheletro.height:,} righe x {scheletro.width} colonne")
scheletro.select(
    "CompanyID", "Year_Delta", "Delta", "OwnershipStatus", "LastKnownValuation", "Revenue"
).filter(pl.col("OwnershipStatus").is_not_null()).head(4)


### 1.10 — selezione delle colonne, filtro e scrittura

`db_master_1` tiene le 29 colonne **invarianti nel tempo** (`R:230-234`): tutto
ciò che varia è già passato nello scheletro.

Poi il filtro `YearFounded > 1999` su entrambi, che porta le aziende a
**116.920**. Attenzione: è `> 1999`, cioè **include il 2000**. La fase 2b e la
fase 4 useranno `> 2000`, che esclude il 2000: è il **bug B1**, il più grave
del catalogo, e si vedrà alla fase 2b.

Di queste 29 colonne, 13 non arriveranno mai a `db_final` (voce X2):
`CompanyLegalName`, `HQPostCode`, `OwnershipStatusDate`, `ParentCompanyID`,
`ActiveInvestors`, `FormerInvestors`, `Website`, `Affiliated`, `N_Parent`,
`N_Sister`, `N_Subsidiary`.


In [ ]:
DB_MASTER_1_COLUMNS = [
    "CompanyID", "CompanyName", "CompanyLegalName", "YearFounded", "Description",
    "Keywords", "HQCity", "HQCountry", "HQPostCode", "OwnershipStatus",
    "OwnershipStatusDate", "ParentCompanyID", "PrimaryIndustrySector",
    "PrimaryIndustryGroup", "ActiveInvestors", "FormerInvestors", "Website",
    "Website_d", "Linkedin", "Facebook", "Twitter", "N_Affiliated", "Affiliated",
    "N_Parent", "N_Sister", "N_Subsidiary", "Has_Parent", "Has_Sister",
    "Has_Subsidiary",
]

db_master_1 = db_master_1.select(DB_MASTER_1_COLUMNS).filter(pl.col("YearFounded") > 1999)
scheletro = scheletro.filter(pl.col("YearFounded") > 1999)

db_master_1.write_parquet(cfg.interim("db_master_1_v1.parquet"))
scheletro.write_parquet(cfg.interim("db_master_2_skeleton.parquet"))

print(f"db_master_1: {db_master_1.height:,} righe x {db_master_1.width} colonne  (attese 116.920)")
print(f"aziende uniche: {db_master_1['CompanyID'].n_unique():,}")
print(f"scheletro  : {scheletro.height:,} righe azienda-anno")
print(f"anno di fondazione: da {db_master_1['YearFounded'].min()} a {db_master_1['YearFounded'].max()}")

del company, affiliati, vita, con_anni
gc.collect()


#### Ispezione: il flag di presenza non è mai `False`

Se qui comparisse `False`, la sostituzione dei segnaposto sarebbe stata
applicata nel punto sbagliato (T1).


In [ ]:
print(db_master_1["Website_d"].value_counts(sort=True))
print(db_master_1["Linkedin"].value_counts(sort=True))


#### Ispezione: i `Delta` negativi del bug B6

Devono essere **245 righe su 106 aziende**: anni-azienda *precedenti* alla
fondazione, generati dalla sequenza decrescente del blocco 1.8.


In [ ]:
negativi = scheletro.filter(pl.col("Delta") < 0)
print(f"righe con Delta < 0: {negativi.height}  su {negativi['CompanyID'].n_unique()} aziende")
negativi.select("CompanyID", "YearFounded", "Year_Delta", "Delta").head(6)


#### Verifica: 28 colonne su 28

`db_master_1` non ha ancora un checkpoint suo — le sei colonne competitor
arrivano alla fase 3a. Le altre 28 sono già definitive e le confrontiamo subito
con il riferimento: aspettare la fine per scoprire un errore nato all'inizio
sarebbe uno spreco. Le sei mancanti si dichiarano **assenti attese**, così non
contano come errore.

Questo confronto legge `db_master_1.csv`, 82 MB: è l'unico che possiamo
permetterci in-process senza sottoprocesso.


In [ ]:
from src.panel.validate import load_reference, verify  # confronto in-process, solo qui

_attese = {
    "SimilarityScoreMean", "SimilarityScoreMax", "N_Competitors",
    "Same_Country", "N_Europe", "N_Outside_Europe",
}
_rapporto = verify(
    db_master_1,
    load_reference(cfg, "db_master_1.csv"),
    key=["CompanyID"],
    name="db_master_1 (parziale, dopo la fase 1)",
    expected_missing=_attese,
    rtol=cfg.rtol,
)
print(_rapporto.render())
del _rapporto, negativi
gc.collect()

# E tutte le colonne contro la base congelata prima della migrazione.
confronta("db_master_1_v1")
confronta("db_master_2_skeleton")


---
## Stadio 2a — la tabella persona-azienda (`db3`) · CHECKPOINT A

**Corrisponde a** `1_Arrange_DB.R:240-524`. Produce 534.851 righe, una per
coppia `(azienda, persona)`, ed è il primo punto con un file di riferimento
dedicato.

1. **Deduplica** `CompanyBoardTeamRelation` per `(azienda, persona)`,
   ricomponendo le righe duplicate.
2. **Aggancia 16 attributi da `Person.csv`** (956 MB, letto proiettando solo le
   colonne che servono) e ne ricava tre indici di esperienza — posizioni,
   incarichi nei board, altri ruoli — standardizzati con `scale(log(x + 1))`.
   La standardizzazione è **globale su tutta la tabella**, non per azienda:
   confrontarla per gruppo darebbe numeri completamente diversi.
3. **Aggancia l'istruzione** da `PersonEducationRelation`: classifica il titolo
   in cinque livelli e il campo di studi in nove aree, poi aggrega per persona
   (il titolo più alto, l'anno di laurea più antico, l'elenco degli atenei).
   Le catene `case_when` dell'R sono a **corto circuito** — il primo match
   vince — quindi l'ordine dei test è vincolante.
4. **Override dal nome della persona**: se il nome contiene `Ph.D`, ` JD` o
   ` MD`, il titolo più alto viene forzato a 5, e negli ultimi due casi si
   accendono anche `Is_Law` e `Is_Med`.
5. **Aggancia `PositionLevel`** da `PersonPositionRelation` e ne ricava
   `IsFounder`.
6. **Imputa la finestra di permanenza** `StartDate`/`EndDate` in quattro
   passaggi, e da lì `DeltaStart`/`DeltaEnd`: in quali anni di vita
   dell'azienda quella persona era presente.


### Trappole

**`if_else` di dplyr restituisce NA se la condizione è NA.** Alla riga 452 la
condizione è `is.na(StartDate) | year(StartDate) < YearFounded`. Per un'azienda
senza `YearFounded` il confronto vale NA, quindi l'intera condizione vale NA, e
`if_else` **cancella una StartDate perfettamente valida**. `pl.when` invece
tratta una condizione nulla come falsa e terrebbe il valore: sono **5.357
righe**, ed erano l'unica divergenza al primo tentativo su questo checkpoint.
La primitiva `rutils.r_if_else` riproduce il comportamento R.

**`PermanenzaMedia` è un solo numero** (bug **B5**). L'R la calcola con un
`summarise` *senza* `group_by`, nonostante il commento dichiari «per ciascuna
CompanyID»: è la permanenza media su tutto il dataset, usata per imputare
l'`EndDate` di chiunque. Misurato: con la media per azienda il `DeltaEnd` medio
passa da 11,57 a 12,04 anni e le righe con dati di team crescono dello 0,84%.

**`Is_Out` resta NA per le aziende non fallite** (bug **B10**). Dopo il left
join, la condizione `Is_Out == FALSE` vale NA e neutralizza una delle
imputazioni di `EndDate`. Un catch-all successivo recupera i casi, quindi il
bug si auto-sana, ma va riprodotto o i valori intermedi divergono.

**`Is_Other` cerca un valore che non esiste** (bug **B2**): confronta `Field`
con `"Other/Unknown"`, mentre `Field` produce `"Other"` e mai
`"Other/Unknown"`. La colonna è quindi falsa su tutte le righe valorizzate.

**`paste` con NA produce la stringa `"NA"`.** `IsFounder` nasce da
`str_detect(paste(FullTitle, PositionLevel, sep = "; "), "Found")`: siccome
`paste` stringifica i mancanti, la stringa non è mai NA e `IsFounder` non è mai
nullo — è sempre `True` o `False`.

**La deduplica ha due anomalie** (bug **B9**): i duplicati si riselezionano
filtrando per `PersonID` invece che per la coppia, e `coalesce(first(x),
last(x))` non vede un valore presente solo in una riga intermedia.


### 2a.1 — leggere il board team e deduplicare le coppie

`CompanyBoardTeamRelation` ha 535.568 righe: una per ogni legame fra una
persona e un'azienda. Ne vogliamo **una per coppia**, e alla fine saranno
534.851 — il numero che il checkpoint A si aspetta.

Le 14 colonne si leggono tutte perché il blocco di deduplica dell'R fa un
`rbind` fra la tabella e un suo aggregato: i due insiemi di colonne devono
combaciare.

Il meccanismo dell'R (righe 248-278) è: prendi le righe duplicate, riaggregale
tenendo per ogni colonna il primo valore non nullo, mettile **davanti** alla
tabella originale, e poi tieni la prima riga di ogni coppia. Così le righe
ricomposte vincono su quelle originali.

**Bug B9, due anomalie nello stesso blocco** (`fix_dup_coalesce`):

1. i duplicati si riselezionano filtrando per **`PersonID`** invece che per la
   coppia `(CompanyID, PersonID)`, quindi entrano nel lavoro di aggregazione
   anche righe che non erano duplicate;
2. `coalesce(first(x), last(x))` guarda solo la prima e l'ultima riga: con tre
   o più duplicati, **un valore presente solo in mezzo va perso**.


In [ ]:
# Tutte e 14 le colonne: il rbind dell'R richiede insiemi combacianti.
BOARD_COLUMNS = [
    "CompanyID", "PersonID", "PersonName", "FullTitle", "IsOnBoard",
    "RepresentingID", "RepresentingName", "RoleOnBoard", "IsCurrent",
    "Location", "StartDate", "EndDate", "RowID", "LastUpdated",
]
COPPIA = ["CompanyID", "PersonID"]

board = as_na(read_raw(cfg, "CompanyBoardTeamRelation", BOARD_COLUMNS), R_NA)
print(f"righe grezze: {board.height:,}   coppie uniche: {board.select(COPPIA).n_unique():,}")

duplicate = board.filter(board.select(COPPIA).is_duplicated())
if cfg.fix_dup_coalesce:
    da_ricomporre = board.join(duplicate.select(COPPIA).unique(), on=COPPIA, how="semi")
else:
    # Bug B9, prima anomalia: filtra per PersonID, non per la coppia.
    da_ricomporre = board.filter(pl.col("PersonID").is_in(duplicate["PersonID"]))
print(f"righe duplicate: {duplicate.height:,}   righe entrate nell'aggregazione: {da_ricomporre.height:,}")

altre = [x for x in BOARD_COLUMNS if x not in COPPIA]
ricomposte = (
    da_ricomporre.sort(["PersonID", "CompanyID"], maintain_order=True)
    .group_by(COPPIA, maintain_order=True)
    # Bug B9, seconda anomalia: coalesce(first, last) salta le righe in mezzo.
    .agg([coalesce_first_last(x).alias(x) for x in altre])
    .select(BOARD_COLUMNS)
)
board = pl.concat([ricomposte, board]).unique(subset=COPPIA, keep="first", maintain_order=True)
print(f"dopo la deduplica: {board.height:,} righe   (attese 534.851)")
del duplicate, da_ricomporre, ricomposte


### 2a.2 — le nove colonne che restano, e le date

Delle 14 se ne tengono 9. `StartDate` e `EndDate` passano da `parse_date_r`
(la stessa regola a due rami della fase 1) e diventeranno la **finestra di
permanenza** di ogni persona: in quali anni di vita dell'azienda quella
persona era presente.

L'ordinamento `(CompanyID, StartDate)` con i nulli in coda è quello dell'R:
qui non cambia nessun risultato, ma lo teniamo per fedeltà.


In [ ]:
db3 = (
    board.select(
        "CompanyID", "PersonID", "PersonName", "FullTitle", "IsOnBoard",
        "RoleOnBoard", "IsCurrent", "StartDate", "EndDate",
    )
    .with_columns(parse_date_r(pl.col(x)).alias(x) for x in ("StartDate", "EndDate"))
    .sort(["CompanyID", "StartDate"], nulls_last=True)
)
del board
gc.collect()

print(f"date valorizzate: StartDate {db3['StartDate'].is_not_null().sum():,}, "
      f"EndDate {db3['EndDate'].is_not_null().sum():,}  su {db3.height:,}")
db3.head(4)


### 2a.3 — i 16 attributi da `Person.csv`

`Person.csv` pesa 956 MB. Lo leggiamo proiettando **solo** le 17 colonne che
servono, altrimenti non ci sta in memoria.

Otto di queste sono conteggi di ruoli — posizioni attuali e passate, incarichi
nei board attuali e passati, ruoli di consulenza, deal e fondi affiliati — e
sono la base degli indici di esperienza del blocco successivo.

> **Voce M0, la più grave del catalogo.** Questi conteggi sono **alla data di
> estrazione**, non all'anno del panel. La finestra di permanenza della persona
> è temporizzata, i suoi attributi no: l'esperienza attribuita al team di una
> startup nel 2010 è quella che quelle persone hanno oggi, comprese le
> posizioni assunte dopo. Vale anche per `Highest_Degree` e `Earliest_Year`, e
> arriva a quattro delle 47 feature dei modelli. Non passa da nessuno dei dieci
> bug censiti: è nel disegno della pipeline.


In [ ]:
PERSON_COLUMNS = [
    "PersonID", "Gender", "Prefix", "City", "PostCode", "Country", "Biography",
    "University_Institution", "RolesCount", "CurrentPositionsCount",
    "FormerPositionsCount", "CurrentBoardSeatsCount", "FormerBoardSeatsCount",
    "CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
    "AffiliatedDealsCount", "NumberOfAffiliatedFunds",
]
# across(CurrentPositionsCount:NumberOfAffiliatedFunds): gli otto conteggi.
COUNT_COLUMNS = PERSON_COLUMNS[9:]

person = as_na(read_raw(cfg, "Person", PERSON_COLUMNS), R_NA_NAN).with_columns(
    to_num(x) for x in [*COUNT_COLUMNS, "RolesCount"]
)
print(f"Person.csv: {person.height:,} persone")

db3 = db3.join(person, on="PersonID", how="left")
del person
gc.collect()

print(f"persone non trovate in Person.csv: {db3['Gender'].is_null().sum():,} righe su {db3.height:,}")


### 2a.4 — gli indici di esperienza

Tre somme e tre standardizzazioni.

`rowSums(..., na.rm = TRUE)` in R su una riga **tutta** mancante restituisce
**0, non NA** (voce T13). Conseguenza da tenere presente: una persona assente
da `Person.csv` ottiene `Positions = 0`, `BoardSeats = 0`, `OtherRoles = 0`, ed
è **indistinguibile** da una persona effettivamente senza posizioni. Non è un
errore di traduzione, è quello che fa l'R.

`scale()` di R usa la deviazione standard **campionaria** (denominatore `n-1`),
mentre numpy e polars usano `n`: `rutils.scale_r` riproduce la prima (T7). E la
standardizzazione è **globale su tutta la tabella**, non per azienda: rifarla
per gruppo darebbe numeri completamente diversi.

`WorkExperienceIndex` è la media dei tre z-score, e `WorkExp_Idx_Mean` che ne
deriva alla fase 2b **è una delle 47 feature dei modelli**.


In [ ]:
def somma_righe(colonne: list[str]) -> pl.Expr:
    """rowSums(..., na.rm = TRUE): una riga tutta mancante fa 0, non nullo."""
    return pl.sum_horizontal([pl.col(x).fill_null(0.0) for x in colonne])


db3 = db3.with_columns(
    somma_righe(COUNT_COLUMNS).alias("RolesCount_Total"),
    somma_righe(["CurrentPositionsCount", "FormerPositionsCount"]).alias("Positions"),
    somma_righe(["CurrentBoardSeatsCount", "FormerBoardSeatsCount"]).alias("BoardSeats"),
    somma_righe(
        ["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
         "AffiliatedDealsCount", "NumberOfAffiliatedFunds"]
    ).alias("OtherRoles"),
).with_columns(
    # scale_r: sd campionaria (n-1), e globale su tutta la tabella.
    scale_r((pl.col("Positions") + 1).log()).alias("Positions_z"),
    scale_r((pl.col("BoardSeats") + 1).log()).alias("BoardSeats_z"),
    scale_r((pl.col("OtherRoles") + 1).log()).alias("OtherRoles_z"),
)
db3 = db3.with_columns(
    pl.mean_horizontal("Positions_z", "BoardSeats_z", "OtherRoles_z").alias("WorkExperienceIndex")
)

db3.select("Positions", "BoardSeats", "OtherRoles", "WorkExperienceIndex").describe()


### 2a.5 — il titolo di studio: cinque livelli

`PersonEducationRelation` ha 1.246.054 righe, una per titolo conseguito.

`DegreeLevel` nasce da una catena di `grepl` case-insensitive. **L'ordine delle
regole è parte della logica** (voce T9): `case_when` in R si ferma al primo
match. Esempio concreto: `"MD"` compare nella prima regola e `"Master"` nella
seconda, quindi un `"MD"` **non arriva mai** a `Master's`. Riordinare le regole
cambia la classificazione.

Nota (T10): il ramo finale cattura tutto, **anche un `Degree` mancante**.
`DegreeLevel` non è quindi mai nullo, e una persona senza titolo dichiarato
riceve `"Other"`, cioè `Highest_Degree = 1`, non un nullo. La guardia
`ifelse(sum(!is.na(DegreeLevel)) > 0, ...)` dell'R è di conseguenza sempre
vera.


In [ ]:
# 1_Arrange_DB.R:326-333 — l'ordine e' vincolante, il primo match vince.
DEGREE_LEVEL_RULES = [
    (r"PhD|Doctor|MD|PsyD|DPhil|DC|DDS|DPT|OD|JD", "PhD/Doctorate"),
    (r"MBA|LLM|Master|MSc|MPhil|MPA|MFA|MEng|MAcc|Graduate", "Master's"),
    (r"Bachelor|BSc|BEng|Laurea|BS|BFA|BCom|degree|Business Program|Undergrad", "Bachelor's"),
    (
        r"Certified|Certificat|A Levels|A-Levels|Diplom|DEA|DESS|Dipl\.-Ing|"
        r"Executive Development Program|Executive Education|Executive Education program|"
        r"Executive Program|First Legal State Exam|Legal Practice Course|Vordiplom",
        "Diploma/Certificate",
    ),
]
# La gerarchia: Highest_Degree e' l'indice 1-based del livello piu' alto.
DEGREE_HIERARCHY = ["Other", "Diploma/Certificate", "Bachelor's", "Master's", "PhD/Doctorate"]

studi = as_na(
    read_raw(
        cfg, "PersonEducationRelation",
        ["PersonID", "Degree", "Major_Concentration", "GraduatingYear", "Institute"],
    ),
    R_NA,
).with_columns(r_case_when(DEGREE_LEVEL_RULES, pl.col("Degree"), pl.lit("Other")).alias("DegreeLevel"))

print(f"{studi.height:,} titoli di studio")
print(studi["DegreeLevel"].value_counts(sort=True))


### 2a.6 — il campo di studi: nove aree

Due passaggi. Prima si **riempie** `Major_Concentration` quando manca,
deducendola dal nome del titolo: un titolo che contiene `law` diventa `"Law"`,
uno che contiene `MBA` diventa `"Business"`, uno che contiene `Medicine`
diventa `"Medicine"`.

Poi `Field` classifica la concentrazione in nove aree, di nuovo con una catena
a corto circuito. Se `Major_Concentration` è ancora nulla, `Field` è nulla: è
il primo ramo del `case_when` dell'R.

Anche qui **l'ordine conta**: `Economics` viene testata prima di
`Engineering`, e `IT and Computer Science` prima di `Law`. Una concentrazione
in «financial engineering» finisce quindi in Economics, non in Engineering.


In [ ]:
# 1_Arrange_DB.R:350-374 — nove aree, ordine vincolante.
FIELD_RULES = [
    (r"business|management|bank|invest|financ|marketing|real estate|account|"
     r"Entrepreneur|commerce|econom|actuarial science|private equity", "Economics"),
    (r"engineer|civil|electric|mechanic|electronic|Operations Research|material|"
     r"logistic|Engeneering", "Engineering"),
    (r"statistic|machine learning|Natural Language|robot|technolog|comput|data|"
     r"informatic|Artificial Intelligence|information science|information systems|"
     r"data science|softwar", "IT and Computer Science"),
    (r"law|tax|justice|forensic|legal|jurisprudence|intellectual property", "Law"),
    (r"medic|nursing|pharmac|health|immunolog|neuroscien|genetic|physio", "Health and Medicine"),
    (r"social science|strateg|sociology|psychology|anthropology|international relations|"
     r"polit|government|geography|polic|international|foreign service|social studies|"
     r"criminology|cognitive science|public affairs|urban planning|social work|"
     r"human resource|leadership|foreign", "Social Sciences"),
    (r"natural|biolog|chemistry|physic|environmental science|math|geology|life science|"
     r"zoology|agriculture", "Natural Sciences"),
    (r"humanit|literature|histor|philosoph|language|linguist|english|spanis|american|"
     r"religion|classic|french|theolog|cultural studies|europe|arts|design|music|"
     r"architecture|journalism|media|public relations|advertising|communic|education|"
     r"early childhood|special education|administration", "Humanities and Arts"),
]

titolo, area = pl.col("Degree"), pl.col("Major_Concentration")
studi = studi.with_columns(
    # Se l'area manca, la si deduce dal nome del titolo.
    pl.when(area.is_null() & titolo.str.contains("(?i)law").fill_null(False)).then(pl.lit("Law"))
    .when(area.is_null() & titolo.str.contains("(?i)MBA").fill_null(False)).then(pl.lit("Business"))
    .when(area.is_null() & titolo.str.contains("(?i)Medicine").fill_null(False)).then(pl.lit("Medicine"))
    .otherwise(area)
    .alias("Major_Concentration")
)
studi = studi.with_columns(
    # Area ancora nulla -> Field nulla: e' il primo ramo del case_when dell'R.
    pl.when(pl.col("Major_Concentration").is_null())
    .then(None)
    .otherwise(r_case_when(FIELD_RULES, pl.col("Major_Concentration"), pl.lit("Other")))
    .alias("Field")
)

print(studi["Field"].value_counts(sort=True))


### 2a.7 — aggregare l'istruzione per persona

Da una riga per titolo a una riga per persona: quali aree ha toccato, l'anno di
laurea più antico, il livello più alto raggiunto, l'elenco degli atenei.

**Bug B2** (`fix_is_other_label`): `Is_Other` confronta `Field` con
`"Other/Unknown"`, un valore che `Field` **non produce mai** — produce
`"Other"`. La colonna è quindi **falsa su tutte le righe valorizzate**, e
inutilizzabile. Non è fra le 47 feature, quindi non ha contaminato i risultati
pubblicati.

Nota (T11): qui `paste(na.omit(Institute), collapse = "; ")` **rimuove** i
valori mancanti. Alla fase 2b l'aggregazione per azienda-anno userà invece
`paste(unique(Institute))`, che li **include come testo `"NA"`** (bug B7). Due
`paste` a poche righe di distanza, con comportamenti opposti.


In [ ]:
# Il valore che ciascun flag cerca in Field.
FIELD_FLAGS = {
    "Is_Eco": "Economics",
    "Is_Eng": "Engineering",
    "Is_Med": "Health and Medicine",
    "Is_Hum": "Humanities and Arts",
    "Is_IT": "IT and Computer Science",
    "Is_Law": "Law",
    "Is_NS": "Natural Sciences",
    "Is_SS": "Social Sciences",
    # Bug B2: Field non produce mai "Other/Unknown", produce "Other".
    "Is_Other": "Other/Unknown",
}

indice_titolo = (
    pl.col("DegreeLevel")
    .replace_strict({nome: i + 1 for i, nome in enumerate(DEGREE_HIERARCHY)}, default=None)
    .cast(pl.Int64)
)
ha_campo = pl.col("Field").is_not_null().sum() > 0

istruzione = studi.group_by("PersonID").agg(
    [
        pl.when(ha_campo).then(pl.col("Field").eq(valore).any()).otherwise(None).alias(flag)
        for flag, valore in FIELD_FLAGS.items()
    ]
    + [
        pl.col("GraduatingYear").cast(pl.Float64, strict=False).min().alias("Earliest_Year"),
        indice_titolo.max().alias("Highest_Degree"),
        # na.omit: qui i mancanti si scartano. Alla fase 2b invece si includono.
        pl.when(pl.col("Institute").is_not_null().sum() > 0)
        .then(pl.col("Institute").drop_nulls().str.join("; "))
        .otherwise(None)
        .alias("Institute"),
    ]
)
db3 = db3.join(istruzione, on="PersonID", how="left")
del studi, istruzione
gc.collect()

print(f"Is_Other vero: {db3['Is_Other'].sum()}   (bug B2: e' sempre 0)")
print(db3["Highest_Degree"].value_counts(sort=True).sort("Highest_Degree"))


### 2a.8 — gli override dal nome della persona

Se il nome contiene `Ph.D`, ` JD` o ` MD`, il livello più alto viene **forzato
a 5** (`PhD/Doctorate`), e negli ultimi due casi si accendono anche `Is_Law` e
`Is_Med`.

Nota (T12): queste tre `grepl` sono **case-sensitive**, a differenza di tutte
le altre del blocco istruzione. `" jd"` minuscolo non viene riconosciuto.


In [ ]:
nome = pl.col("PersonName")
ha_phd = nome.str.contains(r"Ph\.?D").fill_null(False)
ha_jd = nome.str.contains(" JD", literal=True).fill_null(False)  # case-sensitive
ha_md = nome.str.contains(" MD", literal=True).fill_null(False)  # case-sensitive

print(f"nomi con Ph.D: {db3.select(ha_phd).to_series().sum():,}")
print(f"nomi con ' JD': {db3.select(ha_jd).to_series().sum():,}")
print(f"nomi con ' MD': {db3.select(ha_md).to_series().sum():,}")

db3 = db3.with_columns(
    pl.when(ha_phd | ha_jd | ha_md).then(5).otherwise(pl.col("Highest_Degree")).alias("Highest_Degree"),
    pl.when(ha_jd).then(True).otherwise(pl.col("Is_Law")).alias("Is_Law"),
    pl.when(ha_md).then(True).otherwise(pl.col("Is_Med")).alias("Is_Med"),
)


### 2a.9 — `PositionLevel` e `IsFounder`

`PersonPositionRelation` (1.611.914 righe) porta il livello della posizione,
deduplicato per `(EntityID, PersonID)` e agganciato su
`(CompanyID, PersonID)`.

`IsFounder` nasce da `str_detect(paste(FullTitle, PositionLevel, sep = "; "),
"Found")`. Il dettaglio che conta (T8): **`paste` con un valore mancante
produce la stringa `"NA"`**, non un nullo. Quindi la stringa da cercare non è
mai nulla e `IsFounder` **non è mai nullo**: è sempre `True` o `False`. Se lo
si traducesse con una concatenazione che propaga i nulli, si otterrebbero dei
nulli che nell'R non esistono.


In [ ]:
posizioni = as_na(
    read_raw(cfg, "PersonPositionRelation", ["PersonID", "EntityID", "PositionLevel"]),
    R_NA_NAN,
).unique(subset=["EntityID", "PersonID"], keep="first", maintain_order=True)

db3 = db3.join(
    posizioni, left_on=["CompanyID", "PersonID"], right_on=["EntityID", "PersonID"], how="left"
)
del posizioni
gc.collect()

# paste(x, y, sep="; ") rende un mancante la stringa "NA": mai nullo.
qualifica = pl.concat_str(
    [pl.col("FullTitle").fill_null("NA"), pl.col("PositionLevel").fill_null("NA")], separator="; "
)
db3 = db3.with_columns(qualifica.str.contains("(?i)Found").alias("IsFounder"))

print(f"founder: {db3['IsFounder'].sum():,}   IsFounder nullo: {db3['IsFounder'].null_count()}")


### 2a.10 — le aziende fallite

Da `db_master_1` si prendono le aziende con `OwnershipStatus == "Out of
Business"` e la data in cui lo sono diventate, che servirà a chiudere la
permanenza di chi c'era.

**Bug B10** (`fix_is_out_na`): dopo il left join, `Is_Out` è **nullo** — non
`False` — per tutte le aziende ancora in attività. Nel blocco 2a.13 la
condizione `Is_Out == FALSE` vale quindi `NA` e **neutralizza**
un'imputazione. Un catch-all successivo recupera i casi, quindi il bug si
auto-sana, ma i valori intermedi divergono se non lo si riproduce.

Nota: `db_master_1` qui è la versione **filtrata** `YearFounded > 1999`, quindi
per le aziende più vecchie non c'è nessuna informazione di fallimento.


In [ ]:
aziende = pl.read_parquet(cfg.interim("db_master_1_v1.parquet")).select(
    "CompanyID", "OwnershipStatus", "OwnershipStatusDate"
)
fallite = aziende.filter(pl.col("OwnershipStatus") == "Out of Business").with_columns(
    pl.lit(True).alias("Is_Out"),
    pl.col("OwnershipStatusDate").alias("OutDate"),
)
print(f"aziende fuori mercato: {fallite.height:,}")

# Bug B10: il left join lascia Is_Out nullo, non False, per le altre.
db3 = db3.join(fallite, on="CompanyID", how="left")
print(f"righe con Is_Out nullo: {db3['Is_Out'].null_count():,} su {db3.height:,}")
del aziende, fallite


### 2a.11 — `YearFounded`, dalla tabella **non** filtrata

Riga 448 dell'R. Si aggancia `YearFounded` da `db1`, il file che la fase 1 ha
salvato **senza** il filtro `> 1999` (voce T5). Usare `db_master_1` farebbe
sparire le persone delle aziende più vecchie.

Nota (T14): nell'R questo è un *natural join* — `left_join` senza `by=` — che
unisce su tutte le colonne in comune. Qui è solo `CompanyID`, ma è fragile: se
`db3` avesse già una colonna `YearFounded`, il join cambierebbe da solo senza
che nessuno lo scriva.


In [ ]:
db3 = db3.join(pl.read_parquet(cfg.interim("db1.parquet")), on="CompanyID", how="left")
print(f"righe di aziende senza YearFounded: {db3['YearFounded'].is_null().sum():,}")


### 2a.12 — imputare `StartDate` · **la trappola peggiore di tutta la pipeline**

La regola dell'R (riga 452) è: se `StartDate` manca **oppure** è precedente
alla fondazione, mettici il primo gennaio dell'anno di fondazione.

Ma è scritta con `if_else` di dplyr, e **`if_else` restituisce NA quando la
condizione stessa è NA** (voce T6). Per un'azienda senza `YearFounded` il
confronto `year(StartDate) < YearFounded` vale `NA`, quindi l'intera condizione
vale `NA`, e `if_else` **cancella una `StartDate` perfettamente valida**.

`pl.when` fa l'opposto: tratta una condizione nulla come falsa e tiene il
valore. Da qui `rutils.r_if_else`, che riproduce R.

*Misurato: 5.357 righe di `db3`. Tutte le 11.264 righe che appartengono ad
aziende senza `YearFounded` escono da qui con `StartDate` nulla, e con essa
`DeltaStart` e `DeltaEnd` nulli — quindi quelle persone non verranno contate
in nessun anno.* Era l'unica divergenza al primo tentativo sul checkpoint A.


In [ ]:
inizio_fondazione = pl.date(pl.col("YearFounded"), 1, 1)
inizia_prima_della_fondazione = pl.col("StartDate").dt.year() < pl.col("YearFounded")

prima = db3["StartDate"].is_not_null().sum()
db3 = db3.with_columns(
    # r_if_else, non pl.when: con YearFounded assente la condizione e' nulla
    # e R cancella la StartDate invece di tenerla.
    r_if_else(
        pl.col("StartDate").is_null() | inizia_prima_della_fondazione,
        inizio_fondazione,
        pl.col("StartDate"),
    ).alias("StartDate")
)
dopo = db3["StartDate"].is_not_null().sum()
print(f"StartDate valorizzate: {prima:,} -> {dopo:,}")
print(f"di cui cancellate perche' l'azienda non ha YearFounded: "
      f"{db3.filter(pl.col('YearFounded').is_null())['StartDate'].is_null().sum():,}")


### 2a.13 — imputare `EndDate`: i primi due passaggi

**Primo.** Chi risulta ancora in carica (`IsCurrent == "Yes"`) in un'azienda
non fallita riceve come fine `2024-12-31`.

Due cose da sapere. La condizione include `Is_Out == FALSE`, che per il bug
**B10** vale `NA` su tutte le aziende ancora in attività: la regola quindi
**non scatta quasi mai**. E la data `2024-12-31` è una **costante legata al
vintage dell'estrazione** (voce M4): rieseguire la pipeline su un download del
2026 darebbe finestre diverse per le stesse persone.

**Secondo.** Chi era in un'azienda fallita e non ha una fine riceve la data del
fallimento.


In [ ]:
fine_2024 = pl.date(2024, 12, 31)

db3 = db3.with_columns(
    # Is_Out == FALSE e' nullo dove Is_Out e' nullo (bug B10): la regola
    # scatta solo per le aziende effettivamente fuori mercato.
    r_if_else(
        pl.col("EndDate").is_null() & (pl.col("IsCurrent") == "Yes") & ~pl.col("Is_Out"),
        fine_2024,
        pl.col("EndDate"),
    ).alias("EndDate")
).with_columns(
    r_if_else(
        pl.col("EndDate").is_null() & pl.col("Is_Out"),
        pl.col("OutDate"),
        pl.col("EndDate"),
    ).alias("EndDate")
)
print(f"EndDate valorizzate dopo i primi due passaggi: {db3['EndDate'].is_not_null().sum():,}")


### 2a.14 — `PermanenzaMedia` e l'ultimo passaggio

**Bug B5** (`fix_permanenza_media_per_company`). L'R calcola la permanenza
media con un `summarise` **senza `group_by`**, nonostante il commento
dichiari «per ciascuna CompanyID»: è **un unico numero** valido per tutto il
dataset, usato per imputare la fine di chiunque.

*Misurato: con la media per azienda il `DeltaEnd` medio passa da 11,57 a 12,04
anni, le righe con dati di team da 885.142 a 892.534 (+0,84%), `Total_People`
+0,91% e `Total_Founders` +1,23%.* Sistematico ma piccolo.

Poi l'ultimo passaggio, in due rami: chi non è più in carica
(`IsCurrent == "No"`) riceve `anno di inizio + permanenza media`, limitato a
`2024-12-31`; **tutti gli altri** che hanno un inizio ma non una fine ricevono
direttamente `2024-12-31`. È questo secondo ramo il catch-all che auto-sana il
bug B10.

> **Voce M6.** Imputare la fine di una permanenza con una media è
> un'assunzione forte, e decide **in quali anni una persona viene contata**:
> tocca quindi tutte le feature di team, non solo questa colonna.


In [ ]:
durata = pl.col("EndDate").dt.year() - pl.col("StartDate").dt.year()

if cfg.fix_permanenza_media_per_company:
    permanenza = durata.mean().round(0).over("CompanyID")
else:
    # Bug B5: summarise() senza group_by, quindi un solo numero globale.
    permanenza = pl.lit(
        db3.filter(pl.col("EndDate").is_not_null() & pl.col("StartDate").is_not_null())
        .select(durata.mean().round(0))
        .item()
    )

db3 = db3.with_columns(permanenza.alias("_permanenza"))
print(f"permanenza media usata per l'imputazione: {db3['_permanenza'][0]} anni")

fine_imputata = pl.min_horizontal(
    pl.date(pl.col("StartDate").dt.year() + pl.col("_permanenza").cast(pl.Int64), 1, 1), fine_2024
)
db3 = db3.with_columns(
    pl.when(
        pl.col("EndDate").is_null()
        & (pl.col("IsCurrent") == "No")
        & pl.col("StartDate").is_not_null()
        & pl.col("_permanenza").is_not_null()
    )
    .then(fine_imputata)
    # Il catch-all che auto-sana il bug B10.
    .when(pl.col("EndDate").is_null() & pl.col("StartDate").is_not_null())
    .then(fine_2024)
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).drop("_permanenza")

print(f"EndDate valorizzate alla fine: {db3['EndDate'].is_not_null().sum():,} su {db3.height:,}")


### 2a.15 — la finestra in anni di vita dell'azienda

`DeltaStart` e `DeltaEnd` sono la finestra di permanenza espressa in **età
dell'azienda**: è così che la fase 2b saprà in quali righe del panel contare
quella persona.

Prima si sistema l'incoerenza: se la fine precede l'inizio, la fine diventa
l'inizio (una permanenza di durata zero, non negativa).

Poi l'override sui founder: **`DeltaStart = 0`** per chi è riconosciuto come
fondatore.

> **Voce M5.** Questo override **sovrascrive la data di inizio reale**: si
> assume che un founder ci sia dal primo giorno. È un'euristica ragionevole,
> ma cancella un dato che in alcuni casi era presente.


In [ ]:
db3 = db3.with_columns(
    pl.when(pl.col("EndDate") < pl.col("StartDate"))
    .then(pl.col("StartDate"))
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).with_columns(
    (pl.col("StartDate").dt.year() - pl.col("YearFounded")).alias("DeltaStart"),
    (pl.col("EndDate").dt.year() - pl.col("YearFounded")).alias("DeltaEnd"),
)

founder_con_inizio_diverso = db3.filter(
    pl.col("IsFounder") & pl.col("DeltaStart").is_not_null() & (pl.col("DeltaStart") != 0)
).height
print(f"founder la cui DeltaStart reale non era 0: {founder_con_inizio_diverso:,}  (M5: sovrascritta)")

db3 = db3.with_columns(
    pl.when(pl.col("IsFounder")).then(0).otherwise(pl.col("DeltaStart")).alias("DeltaStart")
)
print(f"DeltaEnd medio: {db3['DeltaEnd'].mean():.4f} anni")


### 2a.16 — le 54 colonne di `db3`, e la scrittura

L'ordine delle colonne è quello del riferimento, così il confronto del
checkpoint A è leggibile.

Delle 54, alla fase 5 ne servono 18 per gli attributi del CEO e alla fase 2b
una dozzina per l'aggregazione. Le altre sembrano non servire a nulla (voci
X5-X8): `Prefix`, `City`, `PostCode`, `Country`, `IsOnBoard`, `RoleOnBoard`,
`PositionLevel`, `OwnershipStatus`, `OwnershipStatusDate`, `OutDate`,
`RolesCount`, i tre z-score intermedi, e `Biography` — un campo di testo
grande, letto da `Person.csv` e mai usato.


In [ ]:
DB3_COLUMNS = [
    "CompanyID", "PersonID", "PersonName", "FullTitle", "IsOnBoard", "RoleOnBoard",
    "IsCurrent", "StartDate", "EndDate", "Gender", "Prefix", "City", "PostCode",
    "Country", "Biography", "University_Institution", "RolesCount",
    "CurrentPositionsCount", "FormerPositionsCount", "CurrentBoardSeatsCount",
    "FormerBoardSeatsCount", "CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
    "AffiliatedDealsCount", "NumberOfAffiliatedFunds", "RolesCount_Total", "Positions",
    "BoardSeats", "OtherRoles", "Positions_z", "BoardSeats_z", "OtherRoles_z",
    "WorkExperienceIndex", "Is_Eco", "Is_Eng", "Is_Med", "Is_Hum", "Is_IT", "Is_Law",
    "Is_NS", "Is_SS", "Is_Other", "Earliest_Year", "Highest_Degree", "Institute",
    "PositionLevel", "IsFounder", "OwnershipStatus", "OwnershipStatusDate", "Is_Out",
    "OutDate", "YearFounded", "DeltaStart", "DeltaEnd",
]

db3 = db3.select(DB3_COLUMNS)
db3.write_parquet(cfg.interim("db3.parquet"))
print(f"db3: {db3.height:,} righe x {db3.width} colonne   (attese 534.851)")


#### Ispezione: la `StartDate` cancellata dall'`if_else`

Le righe di aziende senza `YearFounded` devono avere **tutte** `StartDate`
nulla. Se qui comparissero delle date, avremmo tradotto l'`if_else` con un
`when/otherwise` (T6).


In [ ]:
senza_anno = db3.filter(pl.col("YearFounded").is_null())
print(f"righe di aziende senza YearFounded: {senza_anno.height:,}")
print(f"di queste, con StartDate valorizzata: {senza_anno['StartDate'].is_not_null().sum()}  (deve essere 0)")
senza_anno.select("CompanyID", "PersonID", "StartDate", "EndDate", "DeltaStart", "DeltaEnd").head(4)


#### Ispezione: `IsFounder` non è mai nullo, `Is_Other` non è mai vero

Il primo per la voce T8 (`paste` stringifica i mancanti), il secondo per il
bug B2 (`Field` non produce mai `"Other/Unknown"`).


In [ ]:
print(f"IsFounder nullo: {db3['IsFounder'].null_count()}   (deve essere 0)")
print(f"Is_Other vero  : {db3['Is_Other'].sum()}   (deve essere 0, bug B2)")
print(f"Is_Out nullo   : {db3['Is_Out'].null_count():,}   (bug B10: le aziende non fallite)")


#### Confronto con la base congelata

Tutte le 53 colonne contro l'output che il modulo produceva prima della
migrazione, **senza esclusioni**. È il controllo che dimostra che portare il
codice nel notebook non ha cambiato niente.


In [ ]:
confronta("db3")


#### CHECKPOINT A

Il confronto con `db3.csv`, il file salvato dagli script R, allineato per
`(CompanyID, PersonID)`. Da guardare: righe attese contro ottenute, chiavi
presenti da un solo lato, colonne divergenti. Se una colonna divergesse, il
rapporto la nomina e stampa le prime dieci righe con la loro chiave.


In [ ]:
del db3, senza_anno
gc.collect()
verifica("A")


---
## Stadio 2b — le colonne di team, anno per anno

**Corrisponde a** `1_Arrange_DB.R:527-664`.

`db3` dice per quali anni ciascuna persona era in azienda. Questo stadio
espande quell'informazione a una riga per `(azienda, anno)` e aggrega 23
colonne di team: quante persone, quota di donne, quali aree di studio sono
rappresentate, il titolo più alto e quello medio, gli indici di esperienza,
quanti founder.

Poi fa un **full join** con lo scheletro dello stadio 1. È un full join e non
un left join perché il panel del team può contenere anni-azienda che lo
scheletro non ha: il risultato sono le **1.001.625 righe** che i checkpoint C e
D si aspettano.


### Trappole

**Il filtro di fondazione cambia soglia** (bug **B1**, il più grave del
registro). Lo stadio 1 filtra `YearFounded > 1999`; questo stadio filtra
`YearFounded > 2000`, e così fa lo stadio 4 sui deal. Risultato: **l'intera
coorte di aziende fondate nel 2000 arriva nel panel senza nessun dato di
team**, e siccome `preprocessing.py` a valle scarta le righe con
`Total_People` nullo, quella coorte **spa­risce silenziosamente** dal dataset
finale. Il flag è `fix_founding_year_threshold`.

**Le righe fantasma non esistono, e non per fortuna.** L'aggregazione conta le
persone con `.N`, che conterebbe anche la riga vuota prodotta da un join senza
corrispondenze, dando `Total_People = 1` con tutti i campi della persona nulli.
Non succede perché `YearFounded` arriva dal lato persona: per un anno-azienda
che non ha trovato nessuno vale NA, e il filtro `> 2000` elimina la riga prima
che l'aggregazione la veda. È strutturale, non un caso.

**`paste(unique(Institute), collapse = "; ")` include i NA come testo** (bug
**B7**): nel riferimento 390.544 righe hanno un `Institute` che comincia con
`"NA; "`. È cosmetico — non altera il flag «ateneo fra i primi 50», che cerca
nomi di università — ma va riprodotto, e crea un problema di confronto:
l'export del riferimento ha trasformato in null le celle che valevano *solo*
`"NA"`, quindi su quelle il confronto non può distinguere. Il motore di
verifica lo dichiara invece di nasconderlo.

**`max()` su un gruppo tutto-NA dà `-Inf` in R, `mean()` dà `NaN`.** Il ciclo
di sostituzione della riga 650 include anche `"Inf"` e `"-Inf"`, e li converte
in NA veri: è per questo che quel ciclo gira *dopo* l'aggregazione e non prima.


### 2b.1 — espandere il team ad anni-azienda

`db3` dice, per ogni persona, **in quali anni di vita dell'azienda** era
presente (`DeltaStart`, `DeltaEnd`). Questo blocco trasforma quell'informazione
in una riga per `(azienda, anno, persona)`.

Due griglie unite. La prima copre, per ogni azienda, tutti gli anni dal primo
arrivo all'ultima partenza di chiunque. La seconda copre, per ogni persona, gli
anni fra il proprio arrivo e la propria partenza. Un left join della prima
sulla seconda dà una riga per ogni persona presente in quell'anno-azienda.

**È l'unica chiamata a funzione esterna di questa fase**, e per due ragioni:
produce ~9 milioni di righe che dentro una funzione vengono liberate al
ritorno, e il suo contenuto è un `explode` più un join — la sostanza sta nei
blocchi seguenti. Il codice è in `src/panel/expansions.py`.

**Bug B1, il più grave del catalogo** (`fix_founding_year_threshold`). Il
filtro `YearFounded > 2000` è passato **esplicitamente** alla funzione perché
si veda: la fase 1 ha tagliato a `> 1999`, che **include** il 2000, mentre qui
si taglia a `> 2000`, che lo **esclude**. Risultato: l'intera coorte di aziende
fondate nel 2000 entra nel panel **senza nessun dato di team**, e siccome
`preprocessing.py` a valle scarta le righe con `Total_People` nullo, quella
coorte **spa­risce silenziosamente** dal dataset finale.

Lo stesso filtro ha un secondo effetto, non documentato nell'R (voce T16):
`YearFounded` arriva dal lato persona, quindi un anno-azienda che non ha
trovato nessuno ha `YearFounded` nullo e **viene eliminato da questo filtro**.
È il motivo per cui l'aggregazione del blocco 2b.2 non vede mai la riga vuota
del join, che conterebbe come una persona. **Attenzione: correggere B1 rimuove
anche questa protezione.**


In [ ]:
db3 = pl.read_parquet(cfg.interim("db3.parquet"))

# Bug B1: qui l'R taglia a > 2000, la fase 1 aveva tagliato a > 1999.
soglia_fondazione = 1999 if cfg.fix_founding_year_threshold else 2000
print(f"soglia usata in questa fase: YearFounded > {soglia_fondazione}")
print(f"(la fase 1 aveva usato > 1999)")

espanso = expand_team(db3, founding_year_threshold=soglia_fondazione)
print(f"\nrighe (azienda, anno, persona): {espanso.height:,}")
print(f"anni-azienda distinti: {espanso.select('CompanyID', 'Years').n_unique():,}")


### 2b.2 — aggregare le 23 colonne di team

Da una riga per persona a una riga per anno-azienda. Quante persone, quota di
donne, quali aree di studio sono rappresentate, il titolo più alto e quello
medio, gli indici di esperienza, quanti founder.

Tre cose da sapere su come sono scritte.

**Bug B7** (`fix_institute_na_literal`): `paste(unique(Institute), collapse =
"; ")` **include i valori mancanti come testo `"NA"`**. *Misurato: 390.544
righe del riferimento hanno un `Institute` che comincia con `"NA; "`.* È
cosmetico — non altera il flag «ateneo fra i primi 50», che cerca nomi di
università — ma va riprodotto, e crea un problema di confronto: l'export del
riferimento ha trasformato in nullo le celle che valevano **solo** `"NA"`,
quindi su quelle il confronto non può distinguere.

**Voce T15**: `max()` su un gruppo tutto mancante restituisce `-Inf` in R e
`mean()` restituisce `NaN`. Il ciclo di sostituzione NA della riga 650 include
anche `"Inf"` e `"-Inf"` e li converte in nulli veri: **è per questo che quel
ciclo gira dopo l'aggregazione e non prima**. In polars gli aggregati
restituiscono già nullo su un gruppo tutto mancante, quindi resta da sistemare
solo la stringa vuota di `Institute`.

**Voce M7**: `Percent_Females` usa il numero **totale** di persone come
denominatore, quindi conta anche quelle con genere ignoto. Se il genere è
sconosciuto per metà del team, la quota di donne è **diluita verso il basso**
invece di essere calcolata sui soli casi noti. `Percent_Females` è una delle 47
feature dei modelli.

**Voce M8**: founder e chiunque altro pesano uguale. `Total_Founders` li conta,
ma gli indici di esperienza e di istruzione sono medie su **tutti** i membri
del board team.


In [ ]:
FLAG_AREE = ["Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Other", "Is_Law", "Is_IT"]

istituto = pl.col("Institute")
if not cfg.fix_institute_na_literal:
    # Bug B7: paste() rende un istituto mancante il testo "NA".
    istituto = istituto.fill_null("NA")

totale = pl.len()
team = espanso.group_by(["CompanyID", "Years"]).agg(
    totale.alias("Total_People"),
    # M7: denominatore = tutte le persone, anche quelle di genere ignoto.
    (pl.col("Gender").eq("Female").sum() / totale * 100).alias("Percent_Females"),
    # any(x, na.rm = TRUE): un gruppo tutto mancante da' False, non nullo.
    *[pl.col(x).fill_null(False).any().alias(x) for x in FLAG_AREE],
    pl.col("Earliest_Year").mean().alias("Avg_Earliest_Year"),
    pl.col("Highest_Degree").max().alias("Highest_Degree_Max"),
    pl.col("Highest_Degree").mean().alias("Highest_Degree_Mean"),
    istituto.unique(maintain_order=True).str.join("; ").alias("Institute"),
    pl.col("RolesCount_Total").max().alias("RolesCount_Max"),
    pl.col("RolesCount_Total").mean().alias("RolesCount_Mean"),
    pl.col("Positions").mean().alias("Positions"),
    pl.col("BoardSeats").mean().alias("BoardSeats"),
    pl.col("OtherRoles").mean().alias("OtherRoles"),
    pl.col("WorkExperienceIndex").max().alias("WorkExp_Idx_Max"),
    pl.col("WorkExperienceIndex").mean().alias("WorkExp_Idx_Mean"),
    pl.col("IsFounder").sum().alias("Total_Founders"),
)
del espanso, db3
gc.collect()

# T15: in polars max()/mean() su un gruppo tutto mancante danno gia' nullo;
# resta solo la stringa vuota che paste() produce quando non c'e' nessun nome.
team = team.with_columns(
    pl.when(pl.col("Institute") == "").then(None).otherwise(pl.col("Institute")).alias("Institute")
)

print(f"team: {team.height:,} anni-azienda x {team.width} colonne")
print(f"Institute che iniziano con 'NA; ' (bug B7): "
      f"{team['Institute'].str.starts_with('NA; ').sum():,}")


### 2b.3 — il **full** join con lo scheletro

Non un left join: un **full** join (voce T17).

Lo scheletro della fase 1 copre gli anni da `YearFounded` a `MaxYear`; il panel
del team copre gli anni fra il primo arrivo e l'ultima partenza. **I due
insiemi non si contengono**: una persona può essere arrivata prima della
fondazione registrata, o essere rimasta dopo l'ultimo anno con dati. Il full
join tiene entrambi, ed è così che si arriva alle **1.001.625 righe** che i
checkpoint C e D si aspettano — con un left join sarebbero meno.

Poi due sistemazioni per le righe che arrivano dal lato team e che quindi non
hanno le colonne dello scheletro:

- `YearFounded` si riempie **in avanti e poi indietro** dentro ogni azienda
  (`fill(.direction = "downup")`): l'anno di fondazione è lo stesso per tutte
  le righe di un'azienda, basta trovarlo da qualche parte;
- `Year_Delta` si ricostruisce come `YearFounded + Delta` dove manca.


In [ ]:
scheletro = pl.read_parquet(cfg.interim("db_master_2_skeleton.parquet"))
print(f"scheletro (fase 1): {scheletro.height:,} righe")
print(f"team    (fase 2b): {team.height:,} righe")

panel = (
    scheletro.join(
        team.rename({"Years": "Delta"}), on=["CompanyID", "Delta"], how="full", coalesce=True
    )
    .sort(["CompanyID", "Delta"])
    # L'anno di fondazione e' lo stesso per tutta l'azienda: lo si propaga.
    .with_columns(pl.col("YearFounded").fill_null(strategy="forward").over("CompanyID"))
    .with_columns(pl.col("YearFounded").fill_null(strategy="backward").over("CompanyID"))
    .with_columns(pl.col("Year_Delta").fill_null(pl.col("YearFounded") + pl.col("Delta")))
    .sort(["CompanyID", "Delta"])
)
del team, scheletro
gc.collect()

panel.write_parquet(cfg.interim("db_master_2_team.parquet"))
print(f"\npanel: {panel.height:,} righe x {panel.width} colonne   (attese 1.001.625)")
print(f"righe con dati di team: {panel['Total_People'].is_not_null().sum():,}")


#### Ispezione: la coorte 2000 svuotata dal bug B1

Tutte le righe delle aziende fondate nel 2000 devono avere `Total_People`
**nullo**, perché il filtro di questa fase le ha escluse dal calcolo del team.
Le coorti adiacenti no. È il bug B1 visto in faccia.


In [ ]:
for anno in (1999, 2000, 2001, 2002):
    coorte = panel.filter(pl.col("YearFounded") == anno)
    if coorte.height == 0:
        print(f"coorte {anno}: assente dal panel (la fase 1 taglia a > 1999)")
        continue
    quota = 100 * coorte["Total_People"].is_not_null().sum() / coorte.height
    print(f"coorte {anno}: {coorte.height:>7,} righe, con dati di team {quota:5.1f}%")


#### Ispezione: nessuna riga fantasma

Una riga con `Total_People = 1` e tutti gli altri campi di team nulli
significherebbe che la riga vuota del join è stata contata come una persona
(voce T16). Non deve esistere — e non esiste per una ragione strutturale, non
per fortuna: il filtro del blocco 2b.1 le elimina prima.


In [ ]:
fantasma = panel.filter(
    (pl.col("Total_People") == 1)
    & pl.col("Percent_Females").is_null()
    & pl.col("Total_Founders").is_null()
)
print(f"righe fantasma: {fantasma.height}   (deve essere 0)")


#### Confronto con la base congelata

Tutte le colonne del panel del team contro l'output che il modulo produceva.
Le 23 colonne di team sono **definitive**: nessuna fase successiva le riscrive,
quindi da qui in avanti devono restare identiche a ogni riesecuzione.


In [ ]:
del panel, fantasma
gc.collect()
confronta("db_master_2_team")


---
## Stadio 3a — le colonne competitor · CHECKPOINT B

**Corrisponde a** `1_Arrange_DB.R:681-708`.

`CompanySimilarRelation.csv` (836 MB, 1.340.950 righe) elenca per ogni azienda
le aziende simili, con un punteggio di similarità e un flag che dice se sono
considerate concorrenti. Questo stadio lo aggrega in sei colonne per azienda e
completa `db_master_1`.

La relazione è **orientata**: «l'azienda A dichiara B come simile». La
direzione inversa non viene aggiunta, né qui né allo stadio 7.


### Trappole

**Tre aggregati sono chiamati senza `na.rm`.** `mean(SimilarityScore)`,
`max(SimilarityScore)` e `sum(IsCompetitor == "Yes")` diventano NA se *un solo*
valore manca. In polars gli aggregati saltano i null per default, quindi
darebbero un numero dove l'R dà NA. Su questi dati non scatta mai, ma il
comportamento è riprodotto comunque.

**`Same_Country` usa `any()` senza `na.rm`** (bug **B4**): restituisce NA
quando nessun confronto è vero *e* almeno uno è mancante, invece di `False`.
Sono **1.809 aziende, l'1,55% di quelle con un aggregato competitor** — poche,
ma `Same_Country` è una feature dei modelli.

**`N_Europe` e `N_Outside_Europe` non sono simmetriche** (bug **B8**):
`N_Europe` somma su *tutte* le righe, `N_Outside_Europe` solo su quelle con
similarità sopra 90. Non sono quindi due facce dello stesso conteggio.

**Il continente ha bisogno di tre stati, non due.** L'R usa `countrycode()`,
che restituisce NA per un nome che non sa collocare, e NA non è `False`:
`N_Europe` non conta né l'uno né l'altro, ma `N_Outside_Europe` conta solo
`False`. Quattro nomi cadono in quel caso — **Kosovo, Polynesia, Micronesia e
British Indian Ocean Territory** — e trattarli come «non europei» sbagliava
`N_Outside_Europe` su 44 aziende.

Per non dipendere da una libreria che può cambiare classificazione fra due
release, la mappa paese → continente è stata **ricavata dai dati** e congelata
in `src/panel/data/europe.csv`: `scripts/derive_europe_mapping.py` la deduce
per propagazione di vincoli da `N_Europe` e `N_Outside_Europe` del riferimento.
Due nomi restano indeterminati perché ogni riga che li menziona appartiene a
un'azienda fuori da `db_master_1`; è verificato che assegnarli in un modo o
nell'altro non cambia nessun output.


### 3a.1 — leggere le aziende simili e agganciare il proprio paese

`CompanySimilarRelation.csv` pesa 836 MB e ha 1.340.950 righe: per ogni
azienda, l'elenco delle aziende che PitchBook considera simili, con un
punteggio di similarità e un flag che dice se sono anche **concorrenti**.

Ne leggiamo cinque colonne. `SimilarityScore` va portato a numero: arriva come
testo.

Poi si aggancia il **proprio** paese (`HQCountry`) da `db_master_1`, che servirà
al confronto con il paese del concorrente.

> **Voce M9.** `db_master_1` qui è già filtrato `YearFounded > 1999`. Per le
> righe di aziende fuori dal panel `HQCountry` è quindi **nullo**, e il
> confronto «stesso paese» sarà nullo **per costruzione**, non perché il dato
> manchi davvero.

La relazione è **orientata**: «l'azienda A dichiara B come simile». La
direzione inversa non viene aggiunta, né qui né alla fase 7 (voce M24). Se A
dichiara B ma B non dichiara A, allora B conta fra i simili di A e A non conta
fra quelli di B.


In [ ]:
simili = as_na(
    read_raw(
        cfg, "CompanySimilarRelation",
        ["CompanyID", "SimilarCompanyID", "SimilarityScore", "IsCompetitor",
         "SimilarCompanyHQCountry"],
    ),
    R_NA_NAN,
).with_columns(to_num("SimilarityScore"))
print(f"coppie (azienda, azienda simile): {simili.height:,}")

aziende = pl.read_parquet(cfg.interim("db_master_1_v1.parquet"))
simili = simili.join(aziende.select("CompanyID", "HQCountry"), on="CompanyID", how="left")
print(f"righe di aziende fuori dal panel (HQCountry nullo, M9): "
      f"{simili['HQCountry'].is_null().sum():,}")
print(simili["IsCompetitor"].value_counts(sort=True))


### 3a.2 — il continente, e perché servono **tre** stati e non due

L'R usa `countrycode(SimilarCompanyHQCountry, "country.name", "continent")` e
poi `SimilarCompanyIsEurope = continente == "Europe"`.

Il punto che conta (voce **T21**): `countrycode` restituisce **NA** per un nome
che non sa collocare, e in R `NA == "Europe"` vale `NA`. Quindi
`SimilarCompanyIsEurope` è una colonna a **tre valori**: vero, falso, ignoto —
e i due aggregati del blocco 3a.4 trattano l'ignoto in modo **diverso**.
Ridurlo a due stati sbagliava `N_Outside_Europe` su 44 aziende.

Quattro nomi cadono nell'ignoto: **Kosovo**, **Polynesia**, **Micronesia**,
**British Indian Ocean Territory**.

Per non dipendere da una libreria che può cambiare classificazione fra due
release, la mappa paese → continente è stata **ricavata dai dati** e congelata
in `src/panel/data/europe.csv`. `scripts/derive_europe_mapping.py` la deduce
per propagazione di vincoli: conosce `N_Europe` e `N_Outside_Europe` del
riferimento e, azienda per azienda, deduce la classe di ogni paese quando resta
una sola incognita. Due nomi restano indeterminati perché ogni riga che li
menziona appartiene a un'azienda fuori da `db_master_1`; è **verificato** che
assegnarli in un modo o nell'altro non cambia nessun output, e sono fissati
come `countrycode` li classifica.


In [ ]:
europa = load_europe()
print(f"paesi nella mappa: {len(europa)}")
print(f"  in Europa                : {sum(1 for v in europa.values() if v is True)}")
print(f"  fuori Europa             : {sum(1 for v in europa.values() if v is False)}")
print(f"  continente ignoto (T21)  : {[k for k, v in europa.items() if v is None]}")

simili = simili.with_columns(
    # Tre stati: True, False, None. None != False, e il blocco 3a.4 li
    # tratta in modo diverso.
    pl.col("SimilarCompanyHQCountry")
    .replace_strict(europa, default=None, return_dtype=pl.Boolean)
    .alias("_in_europa")
)
print(f"\nrighe per stato del continente:")
print(simili["_in_europa"].value_counts(sort=True))


### 3a.3 — la soglia 90, e i sottoinsiemi che genera

Tre dei sei aggregati usano **solo** le righe con `SimilarityScore > 90`;
gli altri tre usano tutte le righe. La soglia è arbitraria e applicata in modo
incoerente (voce **M10**).

Il dettaglio che genera il bug B4 (voce **T20**): in R il sottoinsieme si
scrive `SimilarCompanyHQCountry[SimilarityScore > 90]`. Se lo score è
**mancante**, l'indice vale `NA` e R **non scarta la riga**: mette un
**elemento NA** nel sottoinsieme. Quell'NA arriva quindi ad `any()`.

Prepariamo perciò due espressioni:

- `nel_sottoinsieme` — le righe che finiscono nel sottoinsieme dell'R: quelle
  con score sopra 90 **e** quelle con score mancante;
- `stesso_paese` — il confronto fra i due paesi, forzato a nullo dove lo score
  è mancante, perché in R quell'elemento è NA indipendentemente dai paesi.


In [ ]:
punteggio = pl.col("SimilarityScore")

simili = simili.with_columns(
    # T20: uno score mancante non scarta la riga, mette un NA nel sottoinsieme.
    (punteggio.is_null() | (punteggio > 90)).alias("_nel_sottoinsieme"),
    # Dove lo score manca l'elemento e' NA a prescindere dai paesi.
    pl.when(punteggio.is_null())
    .then(None)
    .otherwise(pl.col("SimilarCompanyHQCountry") == pl.col("HQCountry"))
    .alias("_stesso_paese"),
)

print(f"righe con score > 90        : {(punteggio > 90).pipe(lambda e: simili.select(e)).to_series().sum():,}")
print(f"righe con score mancante    : {simili['SimilarityScore'].null_count():,}")
print(f"righe nel sottoinsieme (T20): {simili['_nel_sottoinsieme'].sum():,}")


### 3a.4 — le sei colonne, una per una

**`SimilarityScoreMean`, `SimilarityScoreMax`, `N_Competitors`** sono chiamate
**senza `na.rm`** (voce T19): in R un solo valore mancante rende NA l'intero
aggregato. In polars gli aggregati saltano i nulli per default e darebbero un
numero, quindi la condizione va scritta a mano. *Su questi dati non scatta
mai*, ma è riprodotta.

**`Same_Country`** è `any(paese_simile == paese_proprio)` sul sottoinsieme,
**senza `na.rm`**. È il **bug B4** (`fix_same_country_narm`): `any()` senza
`na.rm` restituisce **NA** quando nessun confronto è vero *e* almeno uno è
mancante, invece di `False`. *Misurato: 1.809 aziende.* Poche, **ma
`Same_Country` è una delle 47 feature dei modelli.*

**`N_Europe` e `N_Outside_Europe`** — è il punto in cui si annidano due
problemi diversi, e vale la pena guardarlo in una tabella. Con
`_in_europa` a tre stati e la soglia 90:

| `_in_europa` | score > 90 | conta in `N_Europe`? | conta in `N_Outside_Europe`? |
|---|---|---|---|
| `True` | sì | **sì** | no |
| `True` | no | **sì** | no |
| `False` | sì | no | **sì** |
| `False` | no | no | **no** |
| `None` (ignoto) | sì | no | **no** |
| `None` (ignoto) | no | no | no |

Due letture di questa tabella:

- **Bug B8** (`fix_europe_asymmetry`): `N_Europe` somma su **tutte** le righe,
  `N_Outside_Europe` solo su quelle sopra 90. Non sono due facce dello stesso
  conteggio, e `N_Europe + N_Outside_Europe` non è il numero di simili.
- **Voce T21**: un paese di continente ignoto non conta né da una parte né
  dall'altra. In R viene da `sum(..., na.rm = TRUE)`, che scarta gli NA in
  entrambi i casi; ma poiché `N_Outside_Europe` somma `!IsEurope`, un `False`
  conta e un `NA` no. **Trattare l'ignoto come «non europeo» sbaglierebbe
  `N_Outside_Europe` su 44 aziende.**


In [ ]:
in_europa = pl.col("_in_europa")
verita = pl.col("_stesso_paese").filter(pl.col("_nel_sottoinsieme"))
e_concorrente = pl.col("IsCompetitor") == "Yes"


def senza_narm(aggregato: pl.Expr, colonna: pl.Expr) -> pl.Expr:
    """T19: un aggregato R chiamato senza na.rm. Un solo mancante lo annulla."""
    return pl.when(colonna.is_null().any()).then(None).otherwise(aggregato)


competitor = simili.group_by("CompanyID").agg(
    senza_narm(punteggio.mean(), punteggio).alias("SimilarityScoreMean"),
    senza_narm(punteggio.max(), punteggio).alias("SimilarityScoreMax"),
    senza_narm(e_concorrente.sum(), pl.col("IsCompetitor")).alias("N_Competitors"),
    # Bug B4: any() senza na.rm e' NA quando niente e' vero e qualcosa manca.
    pl.when(verita.fill_null(False).any())
    .then(True)
    .when(verita.is_null().any())
    .then(None)
    .otherwise(False)
    .alias("Same_Country"),
    # N_Europe: conta i True su TUTTE le righe.
    in_europa.fill_null(False).sum().alias("N_Europe"),
    # N_Outside_Europe (bug B8): conta i False, e SOLO sopra 90. L'ignoto
    # (fill_null(True) -> ~True -> False) non conta da nessuna parte (T21).
    (~in_europa.fill_null(True) & (punteggio > 90).fill_null(False))
    .sum()
    .alias("N_Outside_Europe"),
)
del simili
gc.collect()

print(f"aziende con un aggregato competitor: {competitor.height:,}")
print(f"Same_Country nullo (bug B4): {competitor['Same_Country'].null_count():,}")
print(f"\nB8, l'asimmetria: righe in cui N_Europe > N_Outside_Europe: "
      f"{competitor.filter(pl.col('N_Europe') > pl.col('N_Outside_Europe')).height:,}")
competitor.head(4)


### 3a.5 — completare `db_master_1`

Left join sulle 116.920 aziende. Le aziende senza nessuna riga in
`CompanySimilarRelation` restano con le sei colonne nulle.

Con questo `db_master_1` è **completo**: 35 colonne, ed è il file che il
checkpoint B verifica.


In [ ]:
db_master_1 = aziende.join(competitor, on="CompanyID", how="left")
db_master_1.write_parquet(cfg.interim("db_master_1.parquet"))
del aziende, competitor
gc.collect()

print(f"db_master_1: {db_master_1.height:,} righe x {db_master_1.width} colonne")
print(f"senza aggregato competitor: {db_master_1['N_Competitors'].is_null().sum():,} aziende")


#### Confronto con la base congelata, e CHECKPOINT B

Il checkpoint B è anche la prova che l'estrazione grezza di
`CompanySimilarRelation.csv` che stiamo usando **è la stessa su cui girava
l'R**. Servirà ricordarlo alla fase 7, dove le stesse colonne calcolate sul
panel pubblicato **non** coincidono (voce M2).


In [ ]:
del db_master_1
gc.collect()
confronta("db_master_1")
verifica("B")


---
## Stadio 3b — dipendenti, financials, news

**Corrisponde a** `1_Arrange_DB.R:710-813`. Tre tabelle, tre logiche diverse:

- **dipendenti**: `CompanyEmployeeHistoryRelation` ha più rilevazioni per anno.
  L'R ordina per `(azienda, anno, data decrescente)` e tiene la prima riga di
  ogni anno, cioè **l'ultima rilevazione dell'anno**.
- **financials**: stessa deduplica su `PeriodEndDate`, ma i valori **riempiono
  solo le celle già vuote** del panel (un `coalesce`, non una sovrascrittura):
  i bilanci che lo stadio 1 aveva già agganciato da `Company.csv` vincono.
- **news**: conteggio per `(azienda, anno)`, con **0** dove il join non trova
  niente — non NA.

Nota sui dati, non sul codice: questa estrazione contiene **1.858 news su 371
aziende**, e il riferimento ne conta 1.137 su 248 righe di panel. `N_News` è
quindi zero sul 99,98% delle righe. È coerente col riferimento, ma è una
colonna praticamente vuota e va saputo prima di interpretarla come feature.


### 3b.1 — i dipendenti: l'ultima rilevazione dell'anno

`CompanyEmployeeHistoryRelation` ha 600.805 righe e **più rilevazioni per
anno**. L'R ordina per `(azienda, anno, data decrescente)` e tiene la prima
riga di ogni anno, cioè **la rilevazione più recente dell'anno** (voce T22).
Le date mancanti finiscono in coda, quindi perdono.

Poi si aggancia al panel su `(azienda, Year_Delta)`.

Nota (voce T25): `left_join` di dplyr **abbina NA a NA**, polars no. Riguarda
la singola riga con `Year_Delta` nullo dell'azienda `807550-48`: in R una
rilevazione con anno mancante potrebbe agganciarsi a quella riga, qui no.
Effetto su 1 riga.


In [ ]:
def ultima_del_anno(df: pl.DataFrame, colonna_data: str, colonna_anno: str) -> pl.DataFrame:
    """T22: arrange(azienda, anno, desc(data)) + distinct: vince la rilevazione
    piu' recente dell'anno, e le date mancanti vanno in coda."""
    return df.sort(
        ["CompanyID", colonna_anno, colonna_data],
        descending=[False, False, True],
        nulls_last=True,
    ).unique(subset=["CompanyID", colonna_anno], keep="first", maintain_order=True)


dipendenti = (
    as_na(
        read_raw(cfg, "CompanyEmployeeHistoryRelation", ["CompanyID", "Date", "EmployeeCount"]),
        R_NA,
    )
    .with_columns(parse_date_r(pl.col("Date")).alias("Date"), to_num("EmployeeCount"))
    .with_columns(pl.col("Date").dt.year().alias("Year"))
)
print(f"rilevazioni grezze: {dipendenti.height:,}")
dipendenti = ultima_del_anno(dipendenti, "Date", "Year").select("CompanyID", "Year", "EmployeeCount")
print(f"una per (azienda, anno): {dipendenti.height:,}")

panel = pl.read_parquet(cfg.interim("db_master_2_team.parquet")).join(
    dipendenti, left_on=["CompanyID", "Year_Delta"], right_on=["CompanyID", "Year"], how="left"
)
del dipendenti
print(f"righe del panel con EmployeeCount: {panel['EmployeeCount'].is_not_null().sum():,}")


### 3b.2 — i financials **riempiono**, non sovrascrivono

Stessa deduplica, su `PeriodEndDate`.

La differenza sta nell'innesto (voce T23): l'R scrive
`if_else(is.na(Revenue), Revenue_db8, Revenue)`, che è un **coalesce**. I
bilanci che la fase 1 aveva già agganciato da `Company.csv` **vincono**; questa
tabella riempie soltanto le celle rimaste vuote. Sovrascriverle cambierebbe i
valori che il checkpoint C verifica.


In [ ]:
FINANZIARIE = ["Revenue", "GrossProfit", "NetIncome", "EnterpriseValue", "EBITDA", "EBIT"]

bilanci = (
    as_na(
        read_raw(cfg, "CompanyFinancialRelation", ["CompanyID", "PeriodEndDate", *FINANZIARIE]),
        R_NA,
    )
    .with_columns(parse_date_r(pl.col("PeriodEndDate")).alias("Date"))
    .with_columns(
        pl.col("Date").dt.year().alias("Year_Delta"), *[to_num(x) for x in FINANZIARIE]
    )
)
bilanci = ultima_del_anno(bilanci, "Date", "Year_Delta").select(
    "CompanyID", "Year_Delta", *FINANZIARIE
)
print(f"bilanci, uno per (azienda, anno): {bilanci.height:,}")

prima = panel["Revenue"].is_not_null().sum()
panel = panel.join(bilanci, on=["CompanyID", "Year_Delta"], how="left", suffix="_bil").with_columns(
    # T23: coalesce, non sovrascrittura. Vince il valore gia' presente.
    pl.coalesce(pl.col(x), pl.col(f"{x}_bil")).alias(x)
    for x in FINANZIARIE
)
panel = panel.drop([f"{x}_bil" for x in FINANZIARIE])
del bilanci
print(f"Revenue valorizzato: {prima:,} -> {panel['Revenue'].is_not_null().sum():,}")


### 3b.3 — le news: zero, non nullo

Conteggio per `(azienda, anno)`, e dove il join non trova niente il valore è
**0**, non nullo (voce T24): un'azienda senza news in un anno ne ha zero.

> Nota sui dati, non sul codice (voce **X13**). *Misurato: questa estrazione
> contiene **1.858 news su 371 aziende**, e il riferimento ne conta 1.137
> distribuite su 248 righe di panel.* `N_News` è quindi **zero sul 99,98% delle
> righe**: è di fatto una costante e non può portare informazione a un modello.


In [ ]:
news = (
    as_na(read_raw(cfg, "CompanyNewsRelation", ["CompanyID", "PublishDate"]), R_NA)
    .with_columns(parse_date_r(pl.col("PublishDate")).dt.year().alias("Year"))
    .group_by("CompanyID", "Year")
    .agg(pl.len().alias("N_News"))
)
print(f"news: {news['N_News'].sum():,} su {news['CompanyID'].n_unique():,} aziende")

panel = panel.join(
    news, left_on=["CompanyID", "Year_Delta"], right_on=["CompanyID", "Year"], how="left"
).with_columns(pl.col("N_News").fill_null(0))  # T24: zero, non nullo
del news

panel.write_parquet(cfg.interim("db_master_2_relations.parquet"))
print(f"\npanel: {panel.height:,} righe x {panel.width} colonne")
print(f"N_News nullo: {panel['N_News'].null_count()}   (deve essere 0)")
print(f"N_News > 0  : {(panel['N_News'] > 0).sum():,} righe su {panel.height:,}  (X13)")


#### Ispezione: i financials non hanno sovrascritto niente

Confronto di `Revenue` prima e dopo il blocco 3b.2: nessuna cella già
valorizzata deve essere cambiata (T23).


In [ ]:
prima_3b = pl.read_parquet(
    cfg.interim("db_master_2_team.parquet"), columns=["CompanyID", "Year_Delta", "Revenue"]
)
dopo_3b = panel.select("CompanyID", "Year_Delta", "Revenue")
unite = prima_3b.join(dopo_3b, on=["CompanyID", "Year_Delta"], suffix="_dopo")
sovrascritte = unite.filter(
    pl.col("Revenue").is_not_null() & (pl.col("Revenue") != pl.col("Revenue_dopo"))
)
riempite = unite.filter(pl.col("Revenue").is_null() & pl.col("Revenue_dopo").is_not_null())
print(f"celle sovrascritte: {sovrascritte.height}   (deve essere 0)")
print(f"celle riempite    : {riempite.height:,}")
del prima_3b, dopo_3b, unite, sovrascritte, riempite


#### Confronto con la base congelata

`EmployeeCount`, `N_News` e le sei colonne finanziarie sono **definitive**:
nessuna fase successiva le riscrive.


In [ ]:
del panel
gc.collect()
confronta("db_master_2_relations")


---
## Stadio 4 — deal e investitori

**Corrisponde a** `1_Arrange_DB.R:826-1251`. È lo stadio più intricato.

1. **Investitori per deal.** `DealInvestorRelation` unito a `Investor`
   classifica ogni investitore in sette categorie (venture capital, angel,
   acceleratore, corporate, private equity, investitore pubblico, altro) e
   aggrega per `DealID`: quanti investitori nuovi, le loro dimensioni medie,
   quali categorie sono presenti, quante e quali fra i lead investor.
2. **Riparazione delle date dei deal**, in quattro passaggi successivi: dallo
   stato di proprietà per fallimenti e acquisizioni, dall'anno di fondazione
   per il primo round, e infine riempiendo i buchi rimasti con la **media
   arrotondata per eccesso fra l'anno del deal precedente e quello del
   successivo**.
3. **Regola `Zero_Invested`**: i tipi di deal in cui l'importo manca in oltre
   il 90% dei casi ricevono 0 invece di NA; quelli con meno di 200 occorrenze
   vengono accorpati in `"Other"`.
4. **Quindici flag** sul tipo di deal (preseed, seed, early VC, later VC,
   M&A, uscita pubblica, fallimento, debito, private equity, grant, spin-off,
   crowdfunding, acceleratore, angel, altro).
5. **Aggregazione per `(azienda, anno)`** e innesto nel panel, più `TR_D`.


### L'imputazione RandomForest non è stata portata

Alle righe 1037-1119 l'R addestra un `randomForest(ntree = 50)` **senza
`set.seed`** e lo usa per riempire gli importi mancanti dei deal. Non è
riprodotta. Le sette colonne che creava non vengono prodotte, e gli importi
mancanti restano mancanti per l'imputazione che già gira prima del training.

Il commento in cima a `src/panel/stage4_deals.py` (`RF_SUSPENDED`) elenca le
sette colonne una per una, con la riga R in cui nascevano.

Le ragioni, oltre al seed assente che rende il risultato irriproducibile anche
rieseguendo l'R:

- **temporale**: il modello è addestrato su deal di tutti gli anni e imputa un
  deal del 2013 usando pattern del 2020, con `Age` fra i predittori;
- **train/test**: è fittato sull'intero dataset prima di qualsiasi split, e con
  lui la winsorizzazione al 95° percentile e il vincolo al terzo quartile;
- **prossimità al target**: `DealTypeGrouped` è un predittore, ma i tipi di
  deal sono ciò che determina `GrowthStage`, cioè il target.

Quanto pesava, misurato sui dataset pubblicati e non sul panel: nel dataset con
finestra temporale **5.914 righe su 30.300 (19,5%)** avevano un valore
imputato, pari al **29,7% della massa della feature**; senza finestra circa
9.739 aziende su 30.300 (32,1%). Non è un dettaglio: i risultati vanno rifatti.

La regola `Zero_Invested` appartiene alla stessa famiglia di problemi — è
derivata da statistiche globali su tutto il dataset — ma è **mantenuta**,
perché alimenta `TotalRaised`, e dire «questo tipo di deal non dichiara mai
l'importo, quindi è zero» non è un'imputazione modellistica.


### Trappole

**`pmax(year(DealDate), YearFounded)` non ha `na.rm`.** Un deal che non ha mai
ottenuto una data mantiene quindi `Year_Delta = NA`, finisce in un gruppo nullo
e **esce dal panel** al join. `pl.max_horizontal` ignora i null e restituirebbe
l'anno di fondazione, parcheggiando quei deal sull'anno zero dell'azienda e
**inventando deal in 7.335 anni-azienda**. Era la sola divergenza di questo
stadio al primo tentativo.

**Le sei varianti di `TotalRaised` hanno tre semantiche diverse** dello stesso
numero, e la differenza è il punto: `TotalRaised` è `sum(na.rm = TRUE)` (un
importo ignoto vale zero), `TotalRaised_NA` è NA se **tutti** gli importi
mancano, `TotalRaised_any` è NA se **almeno uno** manca. Le tre `_Est`
corrispondenti erano le versioni con l'imputazione.

**Gli aggregati per investitore sono condizionati a `any(InvestorStatus == "New
Investor")` senza `na.rm`**, quindi diventano NA quando nessuno corrisponde e
qualcosa manca. Il blocco dei lead investor filtra su `IsLeadInvestor` ma
conserva la stessa condizione sui nuovi investitori: l'asimmetria è nell'R e va
riprodotta.

**Il filtro `YearFounded > 2000`** torna qui: è sempre il bug B1.


### 4.1 — gli investitori, classificati in sette categorie

`DealInvestorRelation` (506.641 righe) dice **chi** ha investito in **quale**
deal, se era un investitore nuovo o già presente, e se era lead investor.
`Investor` (92.909 righe) porta le caratteristiche di ciascuno.

`PrimaryInvestorType` ha decine di valori e viene ridotto a sette categorie
(`R:861-871`). Tutto ciò che non rientra nella mappa diventa `"Other"`.

Delle colonne selezionate da `Investor` nell'R — `Description`,
`YearFounded`, `HQCity`, `HQCountry`, `MostLikelyFundraisIng` — **nessuna viene
mai usata** (voce X17); qui leggiamo solo quelle che servono.


In [ ]:
# R:861-871 — la mappa a sette categorie. Fuori mappa -> "Other".
INVESTOR_CATEGORY = {
    "Venture Capital": "Venture Capital",
    "Corporate Venture Capital": "Venture Capital",
    "Growth/Expansion": "Venture Capital",
    "Not-For-Profit Venture Capital": "Venture Capital",
    "VC-Backed Company": "Venture Capital",
    "Angel (individual)": "Angel",
    "Angel Group": "Angel",
    "Accelerator/Incubator": "Accelerator",
    "Corporation": "Corporate",
    "Corporate Development": "Corporate",
    "PE/Buyout": "Private Equity",
    "Family Office": "Private Equity",
    "PE-Backed Company": "Private Equity",
    "Holding Company": "Private Equity",
    "Merchant Banking Firm": "Private Equity",
    "Mezzanine": "Private Equity",
    "Secondary Buyer": "Private Equity",
    "Other Private Equity": "Private Equity",
    "Special Purpose Acquisition Company (SPAC)": "Private Equity",
    "Fundless Sponsor": "Private Equity",
    "Government": "Public Investor",
    "University": "Public Investor",
    "Sovereign Wealth Fund": "Public Investor",
    "Mutual Fund": "Public Investor",
}
# I nomi dei flag has_* e il valore di categoria che ciascuno cerca.
CATEGORIE = {
    "Angel": "Angel",
    "Corporate": "Corporate",
    "VentureCapital": "Venture Capital",
    "Accelerator": "Accelerator",
    "PrivateEquity": "Private Equity",
    "PublicInvestor": "Public Investor",
}
NUMERICHE_INVESTITORE = [
    "TotalActivePortfolio", "TotalInvestments", "MedianRoundAmount", "MedianValuation",
]

relazione = as_na(
    read_raw(cfg, "DealInvestorRelation", ["DealID", "InvestorID", "InvestorStatus", "IsLeadInvestor"]),
    R_NA_NAN,
)
investitori = as_na(
    read_raw(
        cfg, "Investor",
        ["InvestorID", "PrimaryInvestorType", *NUMERICHE_INVESTITORE, "PreferredVerticals"],
    ),
    R_NA_NAN,
).with_columns(to_num(x) for x in NUMERICHE_INVESTITORE)

relazione = relazione.join(investitori, on="InvestorID", how="left").with_columns(
    pl.col("PrimaryInvestorType").replace_strict(INVESTOR_CATEGORY, default="Other").alias("InvestorCategory")
)
del investitori

print(f"partecipazioni a deal: {relazione.height:,}   deal distinti: {relazione['DealID'].n_unique():,}")
print(relazione["InvestorCategory"].value_counts(sort=True))
print(relazione["InvestorStatus"].value_counts(sort=True))


### 4.2 — aggregare per deal, e la condizione che li governa quasi tutti

Quasi tutti gli aggregati di questo blocco sono **condizionati** a
`any(InvestorStatus == "New Investor")`, cioè «questo deal ha almeno un
investitore nuovo?».

Il punto (voce **T28**): quell'`any()` è **senza `na.rm`**, quindi vale `NA`
quando nessuno corrisponde *e* almeno uno ha lo stato mancante. E siccome è la
condizione di un `ifelse`, un `NA` lì rende **`NA` l'aggregato intero** — non
`False`, non zero. Da qui la variabile `condizione` scritta a mano con tre
esiti, e `r_if_else` che propaga il nullo.

Il secondo punto (voce **T29**): il blocco dei **lead investor** filtra su
`IsLeadInvestor == "Yes"` ma **conserva la stessa condizione sui nuovi
investitori**. Cioè: `has_Angel_Lead` è nullo se il deal non ha investitori
nuovi, anche se ha un lead investor angel. È un'asimmetria che sta nell'R e
viene riprodotta.


In [ ]:
nuovo = pl.col("InvestorStatus") == "New Investor"
lead = pl.col("IsLeadInvestor") == "Yes"

# T28: any() senza na.rm ha tre esiti, e come condizione di un ifelse
# un NA rende nullo l'aggregato intero.
condizione = (
    pl.when(nuovo.fill_null(False).any())
    .then(True)
    .when(nuovo.is_null().any())
    .then(None)
    .otherwise(False)
)


def media_sui_nuovi(colonna: str) -> pl.Expr:
    return r_if_else(condizione, pl.col(colonna).filter(nuovo.fill_null(False)).mean(), None)


def ha_categoria(categoria: str, maschera: pl.Expr) -> pl.Expr:
    appartiene = pl.col("InvestorCategory").filter(maschera.fill_null(False)) == CATEGORIE[categoria]
    return r_if_else(condizione, appartiene.fill_null(False).any(), None)


per_deal = relazione.group_by("DealID").agg(
    nuovo.fill_null(False).sum().alias("TotalInvestors"),
    media_sui_nuovi("TotalActivePortfolio").alias("MeanTotalActivePortfolio"),
    media_sui_nuovi("TotalInvestments").alias("MeanTotalInvestments"),
    media_sui_nuovi("MedianRoundAmount").alias("MeanMedianRoundAmount"),
    media_sui_nuovi("MedianValuation").alias("MeanMedianValuation"),
    # toString(unique(x)) = paste(collapse = ", ")
    r_if_else(
        condizione,
        pl.col("PreferredVerticals")
        .filter(nuovo.fill_null(False) & pl.col("PreferredVerticals").is_not_null())
        .unique(maintain_order=True)
        .str.join(", "),
        None,
    ).alias("PreferredVerticals"),
    *[ha_categoria(x, nuovo).alias(f"has_{x}") for x in CATEGORIE],
    (nuovo.fill_null(False) & lead.fill_null(False)).sum().alias("LeadInvestorCount"),
    # T29: filtra sui lead, ma conserva la condizione sui nuovi investitori.
    *[ha_categoria(x, lead).alias(f"has_{x}_Lead") for x in CATEGORIE],
)
del relazione
gc.collect()

print(f"deal con almeno un investitore registrato: {per_deal.height:,}")
print(f"has_Angel nullo (T28): {per_deal['has_Angel'].null_count():,}")
per_deal.head(3)


### 4.3 — i deal, e i dati dell'azienda che servono a ripararli

`Deal.csv` ha 385.481 righe. Ne leggiamo 15 colonne e agganciamo cinque
informazioni da `db_master_1`: l'anno di fondazione, lo stato di proprietà con
la sua data, il paese e il settore. Le prime tre servono a **riparare le date
mancanti** nel blocco 4.4.

`db_master_1` è già filtrato `YearFounded > 1999`, quindi per i deal di aziende
più vecchie `YearFounded` è nullo — e il filtro del blocco 4.4 li eliminerà.

`PercentAcquired` è selezionata qui e **mai aggregata** (voce X18).


In [ ]:
COLONNE_DEAL = [
    "CompanyID", "DealID", "DealNo", "DealDate", "DealType", "PercentAcquired",
    "VCRound", "TotalInvestedCapital", "InvestorOwnership", "BusinessStatus",
    "FinancingStatus", "PremoneyValuation", "PostValuation", "CEOPBId", "DealSynopsis",
]
NUMERICHE_DEAL = [
    "TotalInvestedCapital", "InvestorOwnership", "PremoneyValuation",
    "PostValuation", "PercentAcquired",
]

deal = as_na(read_raw(cfg, "Deal", COLONNE_DEAL), R_NA_NAN).with_columns(
    pl.col("DealNo").cast(pl.Int64, strict=False),
    parse_date_r(pl.col("DealDate")).alias("DealDate"),
    *[to_num(x) for x in NUMERICHE_DEAL],
)
print(f"deal grezzi: {deal.height:,}")
print(f"senza DealDate: {deal['DealDate'].is_null().sum():,}")
print(f"senza TotalInvestedCapital: {deal['TotalInvestedCapital'].is_null().sum():,}")

deal = deal.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "YearFounded", "OwnershipStatus", "OwnershipStatusDate",
        "HQCountry", "PrimaryIndustrySector",
    ),
    on="CompanyID",
    how="left",
)
print(f"deal di aziende fuori dal panel (YearFounded nullo): {deal['YearFounded'].is_null().sum():,}")


### 4.4 — riparare le date dei deal: quattro passaggi

**Voce M14.** Quattro regole successive che riempiono una `DealDate` mancante,
in ordine di plausibilità decrescente:

1. **fallimenti** — un deal di tipo bancarotta o «out of business» in
   un'azienda il cui stato è «Out of Business» prende la data di quello stato.
   Plausibile: sono lo stesso evento visto da due tabelle.
2. **acquisizioni** — un `Merger/Acquisition` in un'azienda acquisita prende la
   data dell'acquisizione. Stessa logica.
3. **primo round** — un deal `DealNo == 1` di tipo iniziale prende il **primo
   gennaio dell'anno di fondazione**. È un'assunzione: il primo round non
   avviene necessariamente l'anno della fondazione.
4. **il resto** — un buco fra due deal datati prende la **media arrotondata per
   eccesso** fra l'anno del deal precedente e quello del successivo. Questa è
   un'invenzione: la data non è ricavata da nessun dato, è interpolata.

Fra il secondo e il terzo passaggio c'è il filtro `YearFounded > 2000`, che è
di nuovo il **bug B1** (`fix_founding_year_threshold`): i deal della coorte
2000 vengono esclusi, come lo erano i suoi dati di team.

Il quarto passaggio ha bisogno del deal precedente e successivo **ordinati per
`(CompanyID, DealNo)`**, non per data — l'ordine è quello dichiarato da
PitchBook, non quello cronologico.


In [ ]:
FALLIMENTO = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business"]
ACQUISITA = ["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]
PRIMO_ROUND = [
    "Accelerator/Incubator", "Angel (individual)", "Grant", "Capitalization",
    "Early Stage VC", "Seed Round", "Spin-Off",
]

manca = pl.col("DealDate").is_null()
data_stato = pl.col("OwnershipStatusDate")
prima_1 = deal["DealDate"].is_null().sum()

# 1. fallimenti
deal = deal.with_columns(
    pl.when(
        manca
        & pl.col("DealType").is_in(FALLIMENTO)
        & (pl.col("OwnershipStatus") == "Out of Business")
        & data_stato.is_not_null()
    )
    .then(data_stato)
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
)
# 2. acquisizioni
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealType") == "Merger/Acquisition")
        & pl.col("OwnershipStatus").is_in(ACQUISITA)
        & data_stato.is_not_null()
    )
    .then(data_stato)
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
)
print(f"date mancanti: {prima_1:,} -> {deal['DealDate'].is_null().sum():,} dopo stato di proprieta'")

# Bug B1: la stessa soglia > 2000 della fase 2b, mentre la fase 1 usava > 1999.
soglia_fondazione = 1999 if cfg.fix_founding_year_threshold else 2000
prima_filtro = deal.height
deal = deal.filter(pl.col("YearFounded") > soglia_fondazione)
print(f"deal dopo il filtro YearFounded > {soglia_fondazione}: {prima_filtro:,} -> {deal.height:,}")

# 3. primo round -> primo gennaio dell'anno di fondazione
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(PRIMO_ROUND)
        & (pl.col("DealNo") == 1)
        & pl.col("YearFounded").is_not_null()
    )
    .then(pl.date(pl.col("YearFounded"), 1, 1))
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
)
print(f"date mancanti dopo il primo round: {deal['DealDate'].is_null().sum():,}")

# 4. interpolazione fra il deal precedente e il successivo, per DealNo
deal = deal.sort(["CompanyID", "DealNo"]).with_columns(pl.col("DealDate").dt.year().alias("_anno"))
anno_prec = pl.col("_anno").shift(1).over("CompanyID")
anno_succ = pl.col("_anno").shift(-1).over("CompanyID")
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealNo") > 1)
        & anno_prec.is_not_null()
        & anno_succ.is_not_null()
    )
    .then(pl.date(((anno_prec + anno_succ) / 2).ceil().cast(pl.Int64), 1, 1))
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
).drop("_anno")
print(f"date mancanti alla fine (M14): {deal['DealDate'].is_null().sum():,}")


### 4.5 — collocare il deal in un anno del panel

`Year_Delta = pmax(year(DealDate), YearFounded)` fa due cose insieme.

**Voce M15**: un deal datato **prima** della fondazione viene schiacciato
sull'anno di fondazione. Non viene scartato: viene **spostato nel tempo**.

**Voce T26, la trappola di questa fase**: `pmax` in R è **senza `na.rm`**,
quindi se `year(DealDate)` è nullo il risultato è **nullo**, non
`YearFounded`. Quei deal finiscono in un gruppo con chiave nulla e **escono dal
panel** al join finale. `pl.max_horizontal` ignora i nulli e restituirebbe
l'anno di fondazione, **inventando deal in 7.335 anni-azienda**.

`UndisclosedAmountFlag` cerca nel testo del deal le parole «undisclosed
amount», «raised» o «received». Serviva **solo** a scegliere le righe da
imputare con la RandomForest: con la RF sospesa è **calcolata e mai usata**
(voce X15), e con lei la lettura di `DealSynopsis` (voce X16). La lasciamo per
non cambiare il conteggio delle colonne, ma è peso morto introdotto dalla
sospensione.


In [ ]:
anno_deal = pl.col("DealDate").dt.year()

deal = deal.with_columns(
    # T26: pmax senza na.rm. Se l'anno manca il risultato e' nullo, e il deal
    # esce dal panel. max_horizontal lo metterebbe sull'anno di fondazione.
    pl.when(anno_deal.is_null())
    .then(None)
    .otherwise(pl.max_horizontal(anno_deal, pl.col("YearFounded")))
    .alias("Year_Delta"),
    # X15: serviva solo alla RandomForest, ora sospesa.
    pl.col("DealSynopsis")
    .str.contains("(?i)undisclosed amount|raised|received")
    .fill_null(False)
    .cast(pl.Int64)
    .alias("UndisclosedAmountFlag"),
)

spostati = deal.filter(anno_deal.is_not_null() & (anno_deal < pl.col("YearFounded")))
print(f"deal spostati sull'anno di fondazione (M15): {spostati.height:,}")
print(f"deal senza anno, che escono dal panel (T26): {deal['Year_Delta'].is_null().sum():,}")

deal = deal.join(per_deal, on="DealID", how="left")
del per_deal, spostati
deal = as_na(deal, R_NA_NAN)
print(f"deal con dati sugli investitori: {deal['TotalInvestors'].is_not_null().sum():,} su {deal.height:,}")


### 4.6 — la regola `Zero_Invested`

**Voce M12.** Si guarda, per ogni tipo di deal, **quale quota** ha l'importo
mancante:

- oltre il **90%** mancante → l'importo diventa **0** invece di nullo. La
  logica è «questo tipo di deal non dichiara mai l'importo, quindi è zero»;
- meno di **200** occorrenze e sotto il 90% → il tipo viene accorpato in
  `"Other"`.

È una regola derivata da **statistiche globali su tutto il dataset**, quindi
appartiene alla stessa famiglia di problemi dell'imputazione RandomForest del
blocco 4.7, in versione più mite. È **mantenuta** perché alimenta
`TotalRaised` e perché non è un'imputazione modellistica — ma è una decisione,
non un fatto.

Dettaglio (voce T31): la tabella dei tipi con importo mancante contiene **solo
i tipi che appaiono almeno una volta con l'importo mancante**. Quindi la
seconda regola, quella dei tipi rari accorpati in `"Other"`, non considera
tutti i tipi di deal ma solo quelli.


In [ ]:
per_tipo = (
    deal.group_by("DealType")
    .agg(pl.len().alias("n_totale"), pl.col("TotalInvestedCapital").is_null().sum().alias("n_mancanti"))
    # T31: solo i tipi che appaiono con l'importo mancante.
    .filter(pl.col("n_mancanti") > 0)
    .with_columns((pl.col("n_mancanti") / pl.col("n_totale")).alias("quota_mancante"))
)
tipi_a_zero = per_tipo.filter(pl.col("quota_mancante") > 0.9)["DealType"].to_list()
tipi_altro = per_tipo.filter((pl.col("n_totale") < 200) & (pl.col("quota_mancante") < 0.9))["DealType"].to_list()

print(f"tipi con oltre il 90% di importi mancanti -> zero ({len(tipi_a_zero)}):")
print("  " + ", ".join(sorted(tipi_a_zero)))
print(f"\ntipi rari accorpati in 'Other' ({len(tipi_altro)}):")
print("  " + ", ".join(sorted(tipi_altro)))

deal = deal.with_columns(
    pl.when(pl.col("DealType").is_in(tipi_a_zero))
    .then(pl.lit("Zero_Invested"))
    .when(pl.col("DealType").is_in(tipi_altro))
    .then(pl.lit("Other"))
    .otherwise(pl.col("DealType"))
    .alias("DealTypeGrouped")
).with_columns(
    pl.when(pl.col("TotalInvestedCapital").is_null() & (pl.col("DealTypeGrouped") == "Zero_Invested"))
    .then(0.0)
    .otherwise(pl.col("TotalInvestedCapital"))
    .alias("TotalInvestedCapital")
)
print(f"\nimporti ancora mancanti: {deal['TotalInvestedCapital'].is_null().sum():,}")


### 4.7 — qui girava la RandomForest, e non è stata portata

**Voce M11.** Alle righe 1037-1119 l'R addestra un
`randomForest(TotalInvestedCapital ~ ., ntree = 50)` e lo usa per riempire gli
importi mancanti, sulle righe dove l'importo manca **e**
`UndisclosedAmountFlag == 1` **e** tutti i predittori sono presenti, con la
predizione limitata al terzo quartile del proprio `DealTypeGrouped`.

**Non è riprodotta.** Le sette colonne che creava non esistono:

| colonna | dove nasceva |
|---|---|
| `TotalInvestedCapital_Est` | livello deal, R:1010 — inizializzata a `TotalInvestedCapital` e poi sovrascritta sulle righe imputate |
| `TotalRaised_Est` | `deals_panel`, R:1160 — `sum(na.rm = TRUE)` |
| `TotalRaised_Est_NA` | `deals_panel`, R:1162 — nullo se **tutti** nulli |
| `TotalRaised_Est_any` | `deals_panel`, R:1164 — nullo se **almeno uno** nullo |
| `TotalRaised_Est_cum` | `2_Arrange_Final.R:114` — cumulata |
| `TotalRaised_Est_any_cum` | `2_Arrange_Final.R:113` — cumulata |
| `TotalRaised_Est_NA_cum` | `2_Arrange_Final.R:115` — cumulata |

Le ragioni, oltre al **seed assente** che rende il risultato irriproducibile
anche rieseguendo l'R:

- **temporale** — addestrata su deal di tutti gli anni, imputa un deal del 2013
  usando pattern del 2020, con `Age` fra i predittori;
- **train/test** — fittata sull'intero dataset prima di qualsiasi split, e con
  lei la winsorizzazione al 95° percentile e il vincolo al terzo quartile;
- **prossimità al target** — `DealTypeGrouped` è un predittore, ma i tipi di
  deal sono ciò che determina `GrowthStage`, cioè il target.

*Misurato sui dataset pubblicati, non sul panel: nel dataset con finestra
temporale **5.914 righe su 30.300 (19,5%)** avevano un valore imputato, pari al
**29,7% della massa della feature**; senza finestra circa 9.739 aziende su
30.300 (32,1%).*

**Voce M13, decisione aperta.** Con la RF sospesa, `src/preprocessing.py` non
ha più `TotalRaised_Est`. Va sostituita con `TotalRaised` — che vale 0 dove
l'importo non è dichiarato — oppure con `TotalRaised_NA`, che lì è nulla e
lascia il buco all'imputazione che gira prima del training. Entrambe vengono
prodotte dal blocco 4.9.


In [ ]:
# Le sette colonne che la RandomForest creava, per poterne verificare l'assenza.
RF_SOSPESA = (
    "TotalInvestedCapital_Est",
    "TotalRaised_Est",
    "TotalRaised_Est_NA",
    "TotalRaised_Est_any",
    "TotalRaised_Est_cum",
    "TotalRaised_Est_any_cum",
    "TotalRaised_Est_NA_cum",
)

# Quante righe la RandomForest avrebbe toccato, con i criteri dell'R.
candidate = deal.filter(
    pl.col("TotalInvestedCapital").is_null() & (pl.col("UndisclosedAmountFlag") == 1)
)
print(f"deal che la RandomForest avrebbe imputato (importo mancante e sinossi")
print(f"che parla di importo non dichiarato): {candidate.height:,} su {deal.height:,}")
print(f"\ncolonne _Est presenti in questa tabella: {sorted(set(RF_SOSPESA) & set(deal.columns))}")
del candidate


### 4.8 — i quindici flag sul tipo di deal

Ogni tipo di deal viene mappato su uno o più flag booleani (`R:1122-1151`).
Alcuni tipi ricadono in più flag: `Grant` accende sia `Is_Preseed` che
`Is_Grant`, `Spin-Off` sia `Is_Preseed` che `Is_SpinOff`.

Sono questi flag che, resi cumulativi alla fase 5, determinano `GrowthStage` e
quindi il **target**.


In [ ]:
PRESEED = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Spin-Off",
           "Equity Crowdfunding", "Product Crowdfunding", "Capitalization"]
USCITA_MA = ["Merger/Acquisition", "Buyout/LBO", "Debt - Acquisition", "Debt - Merger",
             "Merger of Equals", "Investor Buyout by Management", "Corporate Asset Purchase",
             "Reverse Merger"]
USCITA_PUBBLICA = ["IPO", "Secondary Transaction - Open Market",
                   "Secondary Transaction - Stock Distribution",
                   "Public Investment 2nd Offering", "PIPE"]
FALLIMENTI = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business",
              "Restart - Angel", "Restart - Early VC", "Restart - Later VC"]
DEBITO = ["Debt - General", "Debt Conversion", "Mezzanine", "Convertible Debt",
          "Debt Refinancing", "Debt - PPP", "Debt Repayment", "Dividend Recapitalization",
          "Exit Financing", "Project Financing", "Share Repurchase", "Leveraged Recapitalization"]
PRIVATE_EQUITY = ["PE Growth/Expansion", "Secondary Transaction - Private", "Corporate",
                  "Platform Creation", "GP Stakes", "General Corporate Purpose", "Capital Spending"]
ALTRI = ["Joint Venture", "Undetermined", "Vendor Loan", "Corporate Licensing",
         "Sale-Lease back facility"]

FLAG_DEAL = {
    "Is_Preseed": PRESEED,
    "Is_Seed": ["Seed Round"],
    "Is_EarlyVC": ["Early Stage VC"],
    "Is_LaterVC": ["Later Stage VC"],
    "Is_MA": USCITA_MA,
    "Is_Public_Exit": USCITA_PUBBLICA,
    "Is_Out": FALLIMENTI,
    "Is_Debt": DEBITO,
    "Is_PE": PRIVATE_EQUITY,
    "Is_Grant": ["Grant"],
    "Is_SpinOff": ["Spin-Off"],
    "Is_CrowdFunding": ["Equity Crowdfunding", "Product Crowdfunding"],
    "Is_Accelerator": ["Accelerator/Incubator"],
    "Is_Angel": ["Angel (individual)"],
    "Other_Deal": ALTRI,
}

deal = deal.with_columns(
    *[pl.col("DealType").is_in(valori).fill_null(False).alias(flag) for flag, valori in FLAG_DEAL.items()]
)
print({flag: int(deal[flag].sum()) for flag in FLAG_DEAL})


### 4.9 — aggregare a `(azienda, anno)`, e le tre semantiche di `TotalRaised`

**Voce T27.** Tre colonne, gli stessi numeri, tre significati diversi del
«mancante» — e la differenza è il punto:

| colonna | come è calcolata | cosa vuol dire un nullo |
|---|---|---|
| `TotalRaised` | `sum(na.rm = TRUE)` | mai nullo: un importo ignoto **vale zero** |
| `TotalRaised_NA` | nullo se **tutti** gli importi mancano | «di nessun deal di quest'anno so l'importo» |
| `TotalRaised_any` | nullo se **almeno uno** manca | «di almeno un deal di quest'anno non so l'importo» |

Le tre `_Est` corrispondenti erano le versioni con l'imputazione e non
esistono più.

Il resto dell'aggregazione: `tail(na.omit(x), 1)` prende **l'ultimo valore non
nullo nell'ordine di riga** per gli stati e le valutazioni; `any(x, na.rm =
TRUE)` per i flag; e due flag — `Is_Accelerator` e `Is_Angel` — sono un **OR**
fra il tipo di deal e la categoria dell'investitore, quindi si accendono anche
se il tipo non lo dice ma l'investitore sì.


In [ ]:
importo = pl.col("TotalInvestedCapital")
vcround = pl.col("VCRound")


def qualunque(colonna: str) -> pl.Expr:
    """any(x, na.rm = TRUE): un gruppo tutto mancante da' False."""
    return pl.col(colonna).fill_null(False).any()


deals_panel = deal.group_by(["CompanyID", "Year_Delta"]).agg(
    pl.len().alias("N_Deal"),
    pl.col("DealType").drop_nulls().unique(maintain_order=True).str.join("; ").alias("DealType"),
    (vcround.is_not_null() & (vcround != "") & (vcround != "Angel")).sum().alias("N_VCround"),
    # T27: tre semantiche del mancante sugli stessi numeri.
    importo.fill_null(0.0).sum().alias("TotalRaised"),
    pl.when(importo.is_null().all()).then(None).otherwise(importo.fill_null(0.0).sum()).alias("TotalRaised_NA"),
    pl.when(importo.is_null().any()).then(None).otherwise(importo.sum()).alias("TotalRaised_any"),
    # tail(na.omit(x), 1): l'ultimo non nullo nell'ordine di riga.
    tail_na_omit("FinancingStatus").alias("FinancingStatus"),
    tail_na_omit("BusinessStatus").alias("BusinessStatus"),
    tail_na_omit("InvestorOwnership").alias("InvestorOwnership"),
    tail_na_omit("PremoneyValuation").alias("PremoneyValuation"),
    tail_na_omit("PostValuation").alias("PostValuation"),
    *[qualunque(f).alias(f) for f in FLAG_DEAL if f not in ("Is_Accelerator", "Is_Angel")],
    # OR fra il tipo di deal e la categoria dell'investitore.
    (qualunque("Is_Accelerator") | qualunque("has_Accelerator")).alias("Is_Accelerator"),
    (qualunque("Is_Angel") | qualunque("has_Angel")).alias("Is_Angel"),
    pl.col("TotalInvestors").fill_null(0).sum().alias("TotalInvestors"),
    pl.col("MeanTotalActivePortfolio").mean().alias("MeanTotalActivePortfolio"),
    pl.col("MeanTotalInvestments").mean().alias("MeanTotalInvestments"),
    pl.col("MeanMedianRoundAmount").mean().alias("MeanMedianRoundAmount"),
    pl.col("MeanMedianValuation").mean().alias("MeanMedianValuation"),
    pl.col("PreferredVerticals").first().alias("PreferredVerticals"),
    *[qualunque(f"has_{x}").alias(f"has_{x}")
      for x in ("Corporate", "VentureCapital", "PrivateEquity", "PublicInvestor")],
    pl.col("LeadInvestorCount").fill_null(0).sum().alias("LeadInvestorCount"),
    *[qualunque(f"has_{x}_Lead").alias(f"has_{x}_Lead") for x in CATEGORIE],
    tail_na_omit("CEOPBId").alias("CEO_ID"),
)
del deal
gc.collect()

deals_panel.write_parquet(cfg.interim("deals_panel.parquet"))
print(f"deals_panel: {deals_panel.height:,} anni-azienda con almeno un deal")
print(f"  di cui nel gruppo ad anno nullo (T26, non si agganciano): "
      f"{deals_panel.filter(pl.col('Year_Delta').is_null()).height:,}")
print(f"TotalRaised nullo    : {deals_panel['TotalRaised'].null_count():,}")
print(f"TotalRaised_NA nullo : {deals_panel['TotalRaised_NA'].null_count():,}")
print(f"TotalRaised_any nullo: {deals_panel['TotalRaised_any'].null_count():,}")


### 4.10 — innestare nel panel, e `TR_D`

`BusinessStatus` del panel viene rinominata `CompanyBusinessStatus` per non
scontrarsi con quella che arriva dai deal. Poi tre `coalesce`: lo stato di
finanziamento, lo stato di business e l'ultima valutazione si **completano** con
i valori dei deal dove il panel non li aveva.

`TR_D` vale 1 quando l'anno-azienda **non ha nessun deal**. Nell'R è calcolata
testando se tutte e **sei** le varianti di `TotalRaised` sono nulle; con tre il
risultato è **identico** (voce T30), perché le `_Est` erano nulle esattamente
dove sono nulle queste. Se un giorno si cambiano le varianti, questa
equivalenza va rivista.

`TR_D` è la colonna che l'R calcola qui e poi **butta via** in
`vars_selected`: noi la teniamo (emendamento E2), perché è il segnale onesto
che distingue «nessun deal» da «importo zero».

L'ultimo passaggio è il ciclo di sostituzione NA con `"Inf"`/`"-Inf"`, che
converte in nulli i `-Inf` dei `max()` e i `NaN` dei `mean()` su gruppi tutti
mancanti.


In [ ]:
panel = pl.read_parquet(cfg.interim("db_master_2_relations.parquet")).rename(
    {"BusinessStatus": "CompanyBusinessStatus"}
)
panel = panel.join(deals_panel, on=["CompanyID", "Year_Delta"], how="left").with_columns(
    pl.coalesce("CompanyFinancingStatus", "FinancingStatus").alias("CompanyFinancingStatus"),
    pl.coalesce("CompanyBusinessStatus", "BusinessStatus").alias("CompanyBusinessStatus"),
    pl.coalesce("LastKnownValuation", "PostValuation").alias("LastKnownValuation"),
    # T30: con tre varianti il risultato e' identico a quello con sei.
    pl.sum_horizontal(pl.col("TotalRaised", "TotalRaised_NA", "TotalRaised_any").is_null())
    .eq(3)
    .cast(pl.Int64)
    .alias("TR_D"),
)
del deals_panel
panel = panel.drop("FinancingStatus", "BusinessStatus", "PostValuation")

# Il ciclo NA finale include "Inf"/"-Inf": converte i -Inf dei max() e i NaN
# dei mean() su gruppi tutti mancanti.
panel = as_na(panel, R_NA_INF)
panel.write_parquet(cfg.interim("db_master_2_deals.parquet"))

print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"TR_D = 1 (anno-azienda senza nessun deal): {(panel['TR_D'] == 1).sum():,}")
print(f"colonne _Est presenti: {sorted(set(RF_SOSPESA) & set(panel.columns))}")


#### Ispezione: i deal senza data non entrano nel panel

Il gruppo con `Year_Delta` nullo di `deals_panel` non deve agganciarsi a
niente, e `TR_D = 1` deve coincidere esattamente con «nessun deal» (T26, T30).


In [ ]:
print(f"righe del panel con Year_Delta nullo e un deal agganciato: "
      f"{panel.filter(pl.col('Year_Delta').is_null() & pl.col('N_Deal').is_not_null()).height}"
      f"   (deve essere 0)")
senza_deal = panel.filter(pl.col("TR_D") == 1)
print(f"righe con TR_D = 1: {senza_deal.height:,}, di cui con N_Deal nullo: "
      f"{senza_deal['N_Deal'].null_count():,}   (devono coincidere)")
del senza_deal


#### Confronto con la base congelata

Cinque colonne sono **definitive** dopo questa fase — `DealType`,
`InvestorOwnership`, `PremoneyValuation`, `PreferredVerticals` e `TR_D` — ma il
confronto le copre tutte, insieme a quelle che la fase 5 renderà cumulative.


In [ ]:
del panel
gc.collect()
confronta("deals_panel")
confronta("db_master_2_deals")


---
## Stadio 5 — finalizzazione · CHECKPOINT C e D

**Corrisponde a tutto** `2_Arrange_Final.R`.

1. **Ventiquattro flag diventano cumulativi.** Prima NA → `False`, poi
   `cumany` per azienda ordinata per anno: una volta che un'azienda ha fatto un
   round seed, il flag resta acceso per sempre. È quello che trasforma «in
   questo anno è successo X» in «a questo anno X era già successo».
2. **`GrowthStage`**, la variabile da cui deriva il target: una cascata di
   sette condizioni a corto circuito, quindi **l'ordine è vincolante**. Fuori
   mercato, uscita pubblica, uscita per M&A, later VC o private equity, early
   VC, seed, preseed; altrimenti NA.
3. Dove `TR_D == 1` le varianti di `TotalRaised` vanno a **0**: se l'anno non
   ha nessun deal, non ha raccolto niente.
4. **Blocco delle cumulate**: numero di deal, di round VC, capitale raccolto,
   investitori, lead investor.
5. **Medie ponderate cumulate** di quattro grandezze degli investitori, pesate
   sul numero di nuovi investitori.
6. **Attributi del CEO**: `CEO_ID` riempito in avanti, poi 18 attributi
   agganciati da `db3`. Sono invarianti nel tempo e cambiano solo quando cambia
   il CEO.
7. **Selezione delle colonne** (`db_selected`), più `StageBlock`,
   `YearsInStage`, `GrowthNextStage` e `TimeNextStage`; infine `db_final`,
   che è `db_selected` più 22 colonne di azienda.


### Trappole

**`cumsum` di R propaga NA fino in fondo al gruppo.** `cumsum(c(1, NA, 3))` è
`c(1, NA, NA)`. Quello di polars lascia un null al suo posto e **continua ad
accumulare**, dando `c(1, null, 4)`. Riguarda `TotalRaised_NA_cum` e
`TotalRaised_any_cum`, cioè proprio le due varianti che diventano NA quando un
importo non è dichiarato: la primitiva `rutils.r_cum_sum` riproduce R.

**`NewInvestors` è calcolato prima che `TotalInvestors` sia sovrascritto** dalla
propria cumulata, dentro lo stesso `mutate`. dplyr valuta in sequenza, e
invertire i due passaggi cambia silenziosamente il peso di tutte le medie
ponderate.

**`rle` tratta ogni NA come una sequenza a sé.** `YearsInStage` nasce da
`sequence(rle(GrowthStage)$lengths)`: due anni consecutivi con stadio mancante
non formano una sequenza di lunghezza due, ma due sequenze di lunghezza uno.

**`StageBlock` diventa NA per l'intera azienda** dal primo `GrowthStage`
mancante in poi (bug **B3**), perché `cumsum` su un confronto che vale NA
propaga. La colonna non è usata a valle.

**La media ponderata cumulata dell'R è O(n²)**; l'equivalente algebrico
`cumsum(x·w) / cumsum(w)` sulle sole righe valide è O(n) ed è **esatto**, non
un'approssimazione.


In [ ]:
stage5_final.run(cfg)
gc.collect()

sel = carica("db_selected", ["CompanyID", "Year_Delta", "Age", "GrowthStage",
                             "GrowthNextStage", "TimeNextStage", "YearsInStage", "N_Deal"])
print(f"db_selected: {sel.height:,} righe   (attese 1.001.625)")
print(f"db_final   : {carica('db_final', ['CompanyID']).height:,} righe (il join non deve moltiplicare)")
print("\ndistribuzione di GrowthStage:")
print(sel["GrowthStage"].value_counts(sort=True))


#### Ispezione: i flag cumulativi non si spengono

Se un'azienda avesse un flag acceso in un anno e spento in quello successivo,
la `cumany` sarebbe stata applicata male (per esempio senza ordinare, o senza
raggruppare per azienda).


In [ ]:
_flag = carica("db_master_2", ["CompanyID", "Year_Delta", "Is_Seed", "Is_EarlyVC", "Is_Out"]).sort(
    ["CompanyID", "Year_Delta"]
)
for _f in ("Is_Seed", "Is_EarlyVC", "Is_Out"):
    _spenti = _flag.filter(pl.col(_f).shift(1).over("CompanyID") & ~pl.col(_f)).height
    print(f"{_f}: righe che si spengono dopo essere state accese = {_spenti}   (deve essere 0)")


#### Ispezione: la cumulata si interrompe al primo importo ignoto

`TotalRaised_any_cum` non deve mai tornare valorizzata dopo essere diventata
nulla, all'interno della stessa azienda.


In [ ]:
_cum = carica("db_master_2", ["CompanyID", "Year_Delta", "TotalRaised_any_cum"]).sort(
    ["CompanyID", "Year_Delta"]
)
_risorte = _cum.filter(
    pl.col("TotalRaised_any_cum").is_null().shift(1).over("CompanyID")
    & pl.col("TotalRaised_any_cum").is_not_null()
).height
print(f"cumulate 'risorte' dopo un nullo: {_risorte}   (deve essere 0)")

del _cum, _flag, sel
gc.collect()


#### CHECKPOINT C e D

`db_master_2` è il panel completo, `db_selected` la sua selezione di colonne.
In entrambi i rapporti le sei colonne `_Est` compaiono come **assenti attese**
(non le produciamo) e in `db_selected` `TR_D` come **in più attesa** (l'R la
calcola e poi la butta, noi la teniamo). Tutto il resto deve coincidere
esattamente.


In [ ]:
verifica("C")


In [ ]:
verifica("D")


---
## Stadio 6 — raggruppamento degli stadi e troncamento · CHECKPOINT E

**Non corrisponde a nessuno script.** Il codice che produceva questo passaggio
è andato perduto: esisteva solo il suo output, `db_master_panel.csv.gz`. Le
quattro regole qui sotto sono state **ricostruite da quel file** e ognuna
verificata contro di esso a divergenza zero su tutte le 882.324 righe. Sono
quindi la specifica, non una congettura.

**R1 — raggruppamento degli stadi.** `GrowthStage` collassa in quattro gruppi,
preservando i nulli: `Preseed`, `Seed` e `EarlyVC` → **Early**;
`LaterVC_or_Other` → **Later**; `Out` → **Out**; `Exit_M&A` e `Exit_Public` →
**Exit**.

**R2 — stadio futuro e distanza, calcolati sulla sequenza NON troncata.** Per
ogni riga si cerca in avanti il primo gruppo diverso e non nullo, e si registra
la distanza in righe. Questo va fatto **prima** del troncamento della regola
R3: dopo, `Out` ed `Exit` non sarebbero più raggiungibili come stadio futuro, e
il senso della colonna è esattamente quello. Se non esiste un gruppo futuro
diverso, il valore è la stringa letterale `"Stay"` e la distanza è quella
dall'**ultima riga non troncata** dell'azienda. Se il gruppo corrente è nullo
non c'è mai corrispondenza, il che riproduce la semantica `NA != x` dell'R.

**R3 — troncamento.** Si eliminano tutte le righe dalla prima riga terminale
(`Out` o `Exit`) in avanti, quella riga compresa. Questo elimina anche le
5.365 righe non terminali che seguono una terminale: non è un effetto
collaterale da aggirare, è quello che deve succedere.

**R4 — `YearsInStage` ricalcolata sul gruppo**, sulla tabella troncata, con la
semantica `rle` di R. Sovrascrive il valore che lo stadio 5 aveva calcolato
sullo stadio non raggruppato.

**`StageBlock` non è riproducibile, ed è dichiarato.** Il valore nel
riferimento non corrisponde né a un ricalcolo su `GrowthStage`, né a uno sul
gruppo, né al valore di `db_selected`: sono 81.954 righe. La colonna è morta —
non la legge né `preprocessing.py` né il resto — quindi la portiamo avanti
invariata e la dichiariamo come divergenza attesa, invece di inseguirla.


In [ ]:
stage6_panel.run(cfg)
gc.collect()

mp = carica("db_master_panel", ["CompanyID", "Year_Delta", "Age", "GrowthStage",
                                "GrowthStageGroup", "GrowthNextStageGroup",
                                "TimeNextStageGroup", "YearsInStage"])
print(f"panel troncato: {mp.height:,} righe   (attese 882.324)")
print(f"aziende: {mp['CompanyID'].n_unique():,}   (attese 116.327)")
print(f"righe eliminate dal troncamento: {1_001_625 - mp.height:,}")
print("\nR1, dallo stadio al gruppo:")
print(mp.group_by("GrowthStage", "GrowthStageGroup").len().sort("len", descending=True))


#### Ispezione: nessuno stadio terminale sopravvive, ma resta raggiungibile

`GrowthStageGroup` non deve mai valere `Out` o `Exit` — quelle righe sono
troncate. `GrowthNextStageGroup` invece sì, e spesso: è la prova che R2 è stata
calcolata prima di R3.


In [ ]:
print("GrowthStageGroup (stadio corrente):")
print(mp["GrowthStageGroup"].value_counts(sort=True))
print("\nGrowthNextStageGroup (stadio futuro):")
print(mp["GrowthNextStageGroup"].value_counts(sort=True))


#### Ispezione: un'azienda passo per passo

Utile per leggere le quattro regole su un caso concreto. `100026-46` entra in
seed, passa a early VC, poi a later VC, e infine esce: la riga dell'uscita e
tutto ciò che segue non ci sono più, ma `GrowthNextStageGroup` la vede.


In [ ]:
mp.filter(pl.col("CompanyID") == "100026-46").sort("Year_Delta").drop("CompanyID")


#### CHECKPOINT E

Il riferimento di questo stadio è l'unico scritto con `write.csv` di R, dove un
valore mancante e la stringa letterale `"NA"` sono gli stessi sei byte e non si
possono distinguere. Il motore di verifica lo sa, lo dichiara, e conta a parte
le celle su cui il confronto è ambiguo.


In [ ]:
del mp
gc.collect()
verifica("E")


---
## Stadio 7 — competitor temporizzati · CHECKPOINT F

**Corrisponde a** `notebook_temporizzazione_competitors.ipynb`, celle 4 e 6.

L'R conta i competitor **una volta sola**, su tutta la lista di aziende simili,
e attacca lo stesso numero a ogni anno del panel. Questo stadio lo sostituisce
con un conteggio dei competitor **effettivamente attivi in ciascun anno**: un
concorrente fondato nel 2015 non può competere nel 2010.

Per ogni azienda si ricava una finestra di vita `[YearFounded, MaxYear]` e un
concorrente conta in un anno solo se quell'anno cade dentro la sua finestra.
Le stesse aggregazioni vengono calcolate anche **senza** il filtro temporale
(le colonne `_All`), per l'esperimento che non usa la finestra.

Infine si sostituisce `"Stay"` con il gruppo corrente, si eliminano `N_Europe`
e `N_Outside_Europe`, e si rimappano i `CompanyID` a interi consecutivi.
**Quest'ultima operazione è l'ultima della pipeline** perché distrugge ogni
possibilità di join con i file di riferimento.


### Tre colonne cambiano significato mantenendo il nome

Va saputo prima di leggerle come se fossero la stessa variabile:

- **`Same_Country`** era un booleano — «esiste un concorrente sopra 90 di
  similarità nel nostro paese» — e diventa un **conteggio** di concorrenti
  attivi nello stesso paese.
- **`SimilarityScoreMean`** viene riempita con **0** dove nessuna azienda
  simile era attiva quell'anno. Zero è il *minimo* della scala, non un valore
  neutro: un'azienda senza concorrenti vivi appare a un modello come
  un'azienda i cui concorrenti sono massimamente diversi.
- **`N_Competitors_All`** non è il `N_Competitors` dell'R: conta solo i
  concorrenti che hanno una finestra di vita utilizzabile, quindi è il minore
  dei due.

Una quarta avvertenza è metodologica e riguarda il paper, non il codice:
`MaxYear` è **l'ultimo anno con dati**, non l'anno in cui l'azienda è morta.
Un'azienda ben coperta da PitchBook risulta quindi viva più a lungo di una
coperta male, e la temporizzazione sovrappesa sistematicamente i concorrenti
grandi.

Una scelta di traduzione: le date passano da `rutils.parse_date_r`, non dal
parser del notebook originale. Il notebook leggeva `%m/%d/%Y` con fallback
`%m/%d/%y`, e chrono interpreta un anno a due cifre `25`-`69` come 2025-2069
mentre R con `cutoff_2000 = 24` lo interpreta come 1925-1969. Su questa
estrazione tutte le date hanno quattro cifre, quindi la differenza è latente e
non attiva; è stata comunque uniformata, perché è la colonna che decide se un
concorrente è vivo e due convenzioni diverse nella stessa pipeline sono un
problema che aspetta di succedere.


In [ ]:
stage7_competitors.run(cfg)
gc.collect()

finale = carica("panel", ["CompanyID", "Year_Delta", "N_Competitors", "Same_Country",
                          "SimilarityScoreMean", "N_Competitors_All",
                          "GrowthStageGroup", "GrowthNextStageGroup"])
print(f"panel finale: {finale.height:,} righe x {carica('panel', None).width} colonne")
print(f"aziende: {finale['CompanyID'].n_unique():,}   CompanyID da {finale['CompanyID'].min()} a {finale['CompanyID'].max()}")
print(f"\n'Stay' residui: {(finale['GrowthNextStageGroup'] == 'Stay').sum()}   (deve essere 0)")
print("\nconcorrenti attivi per anno-azienda:")
print(finale["N_Competitors"].describe())


#### Ispezione: temporizzato contro statico

Il conteggio temporizzato deve essere minore o uguale a quello statico: un
concorrente attivo in un dato anno è per definizione anche un concorrente.


In [ ]:
_conf = finale.select(
    (pl.col("N_Competitors") <= pl.col("N_Competitors_All")).alias("coerente")
)
print(f"righe con temporizzato <= statico: {_conf['coerente'].sum():,} su {finale.height:,}")
print(f"media temporizzata: {finale['N_Competitors'].mean():.2f}")
print(f"media statica     : {finale['N_Competitors_All'].mean():.2f}")


#### CHECKPOINT F — e il suo limite, che è nei dati e non nel codice

107 colonne su 114 riproducono esattamente il panel pubblicato. **Le sei
colonne competitor no, e la causa è accertata**: quel panel le ha calcolate da
un download di `CompanySimilarRelation.csv` diverso da quello che abbiamo.

La prova non è indiziaria:

1. il **checkpoint B** riproduce esattamente tutti gli aggregati competitor di
   `db_master_1.csv` dalla nostra estrazione, quindi la nostra estrazione è
   quella su cui girava l'R;
2. eppure **655.869 righe su 84.138 aziende** hanno lo **stesso numero di
   concorrenti** del panel di riferimento e una **media di similarità diversa**,
   in entrambe le direzioni: stesse aziende, punteggi diversi;
3. sull'azienda `100063-00` la media di riferimento 97,315 **non è la media di
   nessun sottoinsieme** dei suoi dieci punteggi nel nostro file.

I punteggi di similarità sono output di un modello che PitchBook ricalcola fra
un download e l'altro. Non è correggibile dal codice: farli combaciare
significherebbe adattare la traduzione a dati che non abbiamo. Le sei colonne
sono dichiarate in `validate.COMPETITOR_VINTAGE_COLUMNS`, con la motivazione,
e il rapporto le stampa a ogni esecuzione invece di nasconderle.


In [ ]:
verifica("F")


---
## Il registro dei difetti dell'R

Dieci difetti noti, tutti riprodotti per default. Ogni riga della tabella
corrisponde a un flag `fix_*` di `PanelConfig` che vale `False`, dove `False`
significa «comportamento R». Accenderne uno cambia il risultato e quindi rompe
i checkpoint: sono lì per la fase di correzione, non per questa.

| id | flag | cosa succede | impatto misurato |
|---|---|---|---|
| **B1** | `fix_founding_year_threshold` | il panel del team e i deal filtrano `YearFounded > 2000`, il resto `> 1999` | **ALTO.** La coorte 2000 arriva senza dati di team e a valle viene scartata: un'intera coorte di fondazione sparisce in silenzio |
| **B5** | `fix_permanenza_media_per_company` | `PermanenzaMedia` è un `summarise` senza `group_by`, quindi un unico numero globale | BASSO. `DeltaEnd` medio 11,57 contro 12,04 anni; righe con team +0,84%; `Total_Founders` +1,23% |
| **B2** | `fix_is_other_label` | `Is_Other` cerca `"Other/Unknown"`, che `Field` non produce mai | MEDIO. La colonna è falsa su tutte le righe valorizzate: inutilizzabile. Non è fra le feature dei modelli |
| **B4** | `fix_same_country_narm` | `any()` senza `na.rm` dà NA invece di `False` | BASSO come volume — 1.809 aziende, 1,55% — **ma `Same_Country` è una feature dei modelli** |
| **B3** | `fix_stageblock_na` | `cumsum` su un confronto NA propaga: `StageBlock` diventa NA per l'intera azienda | BASSO. La colonna non è usata a valle |
| **B8** | `fix_europe_asymmetry` | `N_Europe` somma su tutte le righe, `N_Outside_Europe` solo sopra 90 di similarità | BASSO. Nessuna delle due è feature dei modelli |
| **B6** | `fix_negative_delta` | se `MaxYear < YearFounded`, `seq()` conta all'indietro e genera `Delta` negativi | TRASCURABILE. 245 righe su 106 aziende |
| **B7** | `fix_institute_na_literal` | `paste(unique(Institute))` include i NA come testo `"NA"` | COSMETICO. Non altera il flag «ateneo fra i primi 50» |
| **B9** | `fix_dup_coalesce` | la deduplica filtra per `PersonID` invece che per la coppia, e `coalesce(first, last)` salta le righe intermedie | BASSO sul risultato |
| **B10** | `fix_is_out_na` | `Is_Out` resta NA per le aziende non fallite e neutralizza due imputazioni di `EndDate` | TRASCURABILE. Un catch-all successivo recupera i casi |


In [ ]:
print("stato dei flag in questa esecuzione:\n")
for _f in BUG_FLAGS:
    _v = getattr(cfg, _f)
    print(f"  {_f:<40} {'CORREZIONE ATTIVA' if _v else 'comportamento R'}")


---
## Riepilogo

| checkpoint | stadio | riferimento | righe | esito |
|---|---|---|---|---|
| A | 2a | `db3.csv` | 534.851 | 52/52 colonne identiche |
| B | 3a | `db_master_1.csv` | 116.920 | 34/34 identiche |
| C | 5 | `db_master_2.csv` | 1.001.625 | 107/107 identiche |
| D | 5 | `db_selected.csv` | 1.001.625 | 89/89 identiche |
| E | 6 | `db_master_panel.csv.gz` | 882.324 | 112/113 identiche |
| F | 7 | `data/raw/panel.csv.gz` | 882.324 | 107/114 identiche |

Tre gruppi di colonne sono **dichiarati** invece che riprodotti, e ogni
esecuzione li stampa:

1. **le sei `TotalRaised_Est*`**, che non produciamo perché l'imputazione
   RandomForest è sospesa (stadio 4);
2. **`StageBlock`** ai checkpoint E e F, non riproducibile e non letta da
   nessuno;
3. **le sei colonne competitor** al checkpoint F, calcolate da un altro
   download della tabella delle aziende simili (stadio 7).

Il panel finito è in `data/interim/panel.csv.gz`. Da lì
`scripts/build_datasets.py` costruisce i due dataset di modellazione.

### Cosa resta da decidere

`src/preprocessing.py` selezionava `TotalRaised_Est`, che non esiste più. Va
sostituita con `TotalRaised` (zero dove l'importo non è dichiarato) oppure con
`TotalRaised_NA` (nulla lì, così l'imputazione che gira prima del training vede
il buco). Entrambe vengono prodotte. In ogni caso i due dataset e tutti i run
vanno rigenerati: la feature del capitale raccolto cambia.

### Per rieseguire tutto senza notebook

```bash
uv run python scripts/check_extraction.py data/raw/pitchbook
uv run python scripts/build_panel.py --verify
```
